# ClimateWell Defence Demo
## Real-time hierarchical climate-policy & well-being evidence mapping

**Purpose.** This notebook consolidates the final logic of the supplied  
`bge-specter-tfidf-ablation.ipynb` and `climatewell-fusenet-improved(1).ipynb` into one defence-ready workflow that:

1. loads and validates the final labelled, master, and audit workbooks;
2. reconstructs the **BGE + TF-IDF classical hierarchical pipeline**;
3. reconstructs **ClimateWell-FuseNet V2**;
4. compares both approaches using training cross-validation and, when verified, the independent audit;
5. selects the operational primary model using a transparent rule;
6. removes every labelled-training and audit record from the demonstration pool;
7. generates **genuine predictions on unseen master-corpus records**; and
8. launches a **Gradio interface** where a title + abstract produces Accept/Reject and, only if accepted, one of the 12 well-being themes.

### Important audit-workbook note
The uploaded `Test_sampled_data.xlsx` uses legacy columns named `pred_decision` and `pred_theme`; it does **not** contain separately named `human_label` columns. Its 450 Reject / 50 Accept distribution matches the final dissertation audit composition, but the notebook will **not silently treat legacy prediction columns as human ground truth**. In the configuration cell, set `AUDIT_LABELS_VERIFIED = True` only after confirming that those two fields are the manually verified audit labels.

If that flag remains `False`, the notebook still runs the complete demo and follows the dissertation's final deployment recommendation: **BGE + TF-IDF as primary model, FuseNet V2 as challenger/disagreement model**.

In [1]:
# ============================================================
# 0. INSTALL DEPENDENCIES (Kaggle / Colab)
# ============================================================
# Run once per fresh Kaggle/Colab runtime. Internet must be ON the first
# time Hugging Face models are downloaded.

import sys, subprocess, importlib.util

packages = {
    "sentence_transformers": "sentence-transformers>=3.0,<6",
    "transformers": "transformers>=4.45,<5",
    "gradio": "gradio>=5,<7",
    "openpyxl": "openpyxl>=3.1",
    "joblib": "joblib>=1.3",
}

missing = [spec for mod, spec in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Required packages are already available.")

print("Dependency check complete.")

Required packages are already available.
Dependency check complete.


In [2]:
# ============================================================
# 1. IMPORTS, PLATFORM DETECTION, REPRODUCIBILITY, CONFIG
# ============================================================
import os, re, gc, json, math, random, warnings, hashlib, zipfile
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.special import expit, softmax

from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import ComplementNB
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.decomposition import TruncatedSVD
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score, f1_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, accuracy_score
)

import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IS_KAGGLE = Path("/kaggle").exists()
IS_COLAB = "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Kaggle:", IS_KAGGLE, "| Colab:", IS_COLAB)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
else:
    print("WARNING: FuseNet training is intended for a GPU runtime.")

# -------------------- USER SETTINGS --------------------
QUICK_MODE = False
# False = defence/final run: 5 Stage-1 folds, up to 70 FuseNet epochs.
# True  = smoke test only: 3 folds, fewer epochs. Do NOT quote QUICK_MODE
# metrics in the dissertation or defence.

AUDIT_LABELS_VERIFIED = False
# Set True ONLY if you have confirmed that Test_sampled_data.xlsx:
#   pred_decision = manually verified external-audit Accept/Reject label
#   pred_theme    = manually verified theme for accepted audit records

AUDIT_DECISION_COL = "pred_decision"
AUDIT_THEME_COL = "pred_theme"

RUN_FULL_MASTER_INFERENCE = True
LAUNCH_GRADIO_SHARE_LINK = True   # useful in Colab/Kaggle
SAVE_MODEL_ARTIFACTS = True

N_FOLDS = 3 if QUICK_MODE else 5
MAX_EPOCHS = 25 if QUICK_MODE else 70
PATIENCE = 5 if QUICK_MODE else 10
BATCH_SIZE = 32
EMBED_BATCH_SIZE = 32 if torch.cuda.is_available() else 8

# Models used in the dissertation
BGE_MODEL_NAME = "BAAI/bge-small-en-v1.5"
SPECTER_MODEL_NAME = "allenai/specter"

# Classical BGE + TF-IDF
WORD_MAX_FEATURES = 80_000
CHAR_MAX_FEATURES = 50_000
EMBEDDING_WEIGHT = 2.0

# FuseNet V2
SVD_DIM = 256
HIDDEN_DIM = 256
DROPOUT = 0.25
BRANCH_DROPOUT_PROB = 0.12
GATE_FLOOR = 0.05
GATE_ENTROPY_WEIGHT = 0.010
GATE_BALANCE_WEIGHT = 0.004
THEME_LOSS_WEIGHT = 1.75
PROTO_LOGIT_WEIGHT = 0.20
PROTO_TEMPERATURE = 0.10
THEME_LABEL_SMOOTHING = 0.04
BINARY_FOCAL_GAMMA = 1.0
THEME_FOCAL_GAMMA = 0.7
LR = 1.2e-3
WEIGHT_DECAY = 2e-4

ROOT = Path("/kaggle/working") if IS_KAGGLE else Path("/content") if IS_COLAB else Path.cwd()
OUTPUT_DIR = ROOT / "climatewell_defence_demo"
CACHE_DIR = OUTPUT_DIR / "cache"
ARTIFACT_DIR = OUTPUT_DIR / "artifacts"
for p in [OUTPUT_DIR, CACHE_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Outputs:", OUTPUT_DIR)

Kaggle: True | Colab: True
Device: cuda
GPU: Tesla T4
Compute capability: (7, 5)
Outputs: /kaggle/working/climatewell_defence_demo


## Kaggle / Colab setup

### Kaggle
1. Create a new notebook.
2. **Accelerator → GPU** (P100/T4 or better).
3. **Internet → On** for the first model download.
4. Add the three Excel files as a Kaggle Dataset or upload them to the notebook input.
5. Upload this notebook and run all cells.

### Google Colab
1. **Runtime → Change runtime type → GPU**.
2. Upload the three Excel files into `/content`, or mount Google Drive.
3. Run all cells.

The code below searches Kaggle input folders, `/content`, the current folder, and `/mnt/data` automatically.

In [3]:
# ============================================================
# 2. FIND THE THREE DATA FILES AUTOMATICALLY
# ============================================================
REQUIRED_FILES = {
    "train": "Human labelled_DTU.xlsx",
    "master": "Master file_10k papers.xlsx",
    "audit": "Test_sampled_data.xlsx",
}

def locate_file(filename):
    direct = [
        Path(filename),
        Path.cwd() / filename,
        Path("/content") / filename,
        Path("/mnt/data") / filename,
    ]
    for p in direct:
        if p.exists():
            return p.resolve()

    roots = [Path("/kaggle/input"), Path("/content/drive/MyDrive")]
    for root in roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    return None

FILES = {k: locate_file(v) for k, v in REQUIRED_FILES.items()}

missing = [REQUIRED_FILES[k] for k, p in FILES.items() if p is None]
if missing and IS_COLAB:
    print("Missing files in Colab:", missing)
    print("A file picker will open. Upload the missing files.")
    from google.colab import files as colab_files
    colab_files.upload()
    FILES = {k: locate_file(v) for k, v in REQUIRED_FILES.items()}
    missing = [REQUIRED_FILES[k] for k, p in FILES.items() if p is None]

if missing:
    raise FileNotFoundError(
        "Could not find: " + ", ".join(missing) +
        "\nUpload them to Kaggle/Colab, then rerun this cell."
    )

for k, p in FILES.items():
    print(f"{k:>6}: {p}")

 train: /kaggle/input/datasets/hrithikmajumdar/climate-text-dataset/Human labelled_DTU.xlsx
master: /kaggle/input/datasets/hrithikmajumdar/climate-text-dataset/Master file_10k papers.xlsx
 audit: /kaggle/input/datasets/hrithikmajumdar/climate-text-dataset/Test_sampled_data.xlsx


In [5]:
# ============================================================
# 3. CLEANING, LABEL NORMALISATION, AND DATA INTEGRITY
#    ROBUST VERSION FOR THE ACTUAL UPLOADED WORKBOOKS
# ============================================================

import re
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# A. BASIC NORMALISATION HELPERS
# ------------------------------------------------------------

def clean_text(x):
    """Convert a cell to clean single-line text."""
    if pd.isna(x):
        return ""

    x = str(x)

    # Normalise common punctuation
    x = (
        x.replace("\u2013", "-")
         .replace("\u2014", "-")
         .replace("\u2212", "-")
         .replace("\ufeff", "")
         .replace("\xa0", " ")
    )

    x = re.sub(r"\s+", " ", x).strip()
    return x


def norm_key(x):
    """
    Normalised form used for robust matching of:
    - column names
    - decision labels
    - theme labels
    """
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        clean_text(x).lower()
    ).strip()


def norm_id(x):
    """Normalise document IDs for overlap checking."""
    return re.sub(
        r"\s+",
        "",
        clean_text(x).lower()
    )


# ------------------------------------------------------------
# B. ROBUST EXCEL SHEET / HEADER READER
# ------------------------------------------------------------

def choose_sheet(path, preferred_sheets=None):
    """
    Select the intended data sheet instead of accidentally loading
    helper sheets such as 'Themes'.
    """
    xls = pd.ExcelFile(path)

    if preferred_sheets:
        normalised = {
            norm_key(sheet): sheet
            for sheet in xls.sheet_names
        }

        for preferred in preferred_sheets:
            key = norm_key(preferred)
            if key in normalised:
                return normalised[key]

    # Avoid auxiliary theme sheets when possible
    for sheet in xls.sheet_names:
        if norm_key(sheet) not in {"theme", "themes"}:
            return sheet

    return xls.sheet_names[0]


def read_excel_smart(
    path,
    preferred_sheets=None,
    expected_headers=None,
    scan_rows=10
):
    """
    Automatically finds the real header row.

    This protects against:
    - header rows shifted down by one row
    - extra blank rows
    - worksheet title rows
    - spaces/newlines in Excel headers
    """

    sheet = choose_sheet(path, preferred_sheets)

    preview = pd.read_excel(
        path,
        sheet_name=sheet,
        header=None,
        nrows=scan_rows
    )

    expected_norm = set(
        norm_key(x)
        for x in (expected_headers or [])
    )

    best_row = 0
    best_score = -1

    for row_idx in range(len(preview)):
        row_values = {
            norm_key(v)
            for v in preview.iloc[row_idx].tolist()
            if clean_text(v)
        }

        score = len(row_values & expected_norm)

        if score > best_score:
            best_score = score
            best_row = row_idx

    # If nothing useful was identified, normal Excel header=0
    # remains the fallback.
    if best_score <= 0:
        best_row = 0

    df = pd.read_excel(
        path,
        sheet_name=sheet,
        header=best_row
    )

    # Clean actual pandas column names
    df.columns = [
        clean_text(c)
        for c in df.columns
    ]

    print(
        f"Loaded: {path}\n"
        f"  sheet      = {sheet!r}\n"
        f"  header row = {best_row}\n"
        f"  rows       = {len(df):,}\n"
        f"  columns    = {list(df.columns)}\n"
    )

    return df


def find_column(df, candidates, required=True):
    """
    Find a dataframe column using robust aliases.

    Example:
        'Article ID'
        'Article_ID'
        'article id'

    are treated equivalently.
    """

    lookup = {
        norm_key(col): col
        for col in df.columns
    }

    # First try exact normalised aliases
    for candidate in candidates:
        key = norm_key(candidate)

        if key in lookup:
            return lookup[key]

    # Then try conservative partial matching
    for candidate in candidates:
        candidate_key = norm_key(candidate)

        for key, original in lookup.items():

            if (
                candidate_key
                and (
                    candidate_key == key
                    or candidate_key in key
                    or key in candidate_key
                )
            ):
                return original

    if required:
        raise KeyError(
            "\nCould not locate any of these columns:\n"
            f"{candidates}\n\n"
            "Columns actually found in workbook:\n"
            f"{list(df.columns)}"
        )

    return None


# ------------------------------------------------------------
# C. CANONICAL THEMES
# ------------------------------------------------------------

CANONICAL_THEMES = [
    "Access to Services and Wellbeing",
    "Better Well-being Metric",
    "Climate change mitigation options & Well-being",
    "Ecosystem & Non-human Well-being",
    "Empowering People & Governance, Policy",
    "Equity, Social Justice, Just Transition",
    "Implemented mitigation Options and Well-being",
    "Integrated Responses and Well-being",
    "SDGs and Well-being",
    "Systems Thinking- Modelling and Supply Chain",
    "Well-being Loss from Climate Mitigation",
    "Well-being, Community & Societal Systems Thinking",
]

THEME_NORM_MAP = {
    norm_key(theme): theme
    for theme in CANONICAL_THEMES
}


# ------------------------------------------------------------
# D. BINARY LABEL NORMALISATION
# ------------------------------------------------------------

INVALID_DECISION_KEYS = {
    "",
    "accept reject",
    "accept or reject",
    "accepted rejected",
    "decision",
    "label",
    "screening",
    "include exclude",
    "yes no",
    "duplicate",          # IMPORTANT: 15 raw duplicate rows
    "duplicated",
    "nan",
    "none",
    "na",
}


def decision_to_binary(x):
    k = norm_key(x)

    if k in INVALID_DECISION_KEYS:
        return np.nan

    if k in {
        "accept",
        "accepted",
        "include",
        "included",
        "relevant",
        "yes",
        "1",
        "true",
    }:
        return 1

    if k in {
        "reject",
        "rejected",
        "exclude",
        "excluded",
        "irrelevant",
        "no",
        "0",
        "false",
    }:
        return 0

    has_accept = any(
        s in k
        for s in ["accept", "include", "relevant"]
    )

    has_reject = any(
        s in k
        for s in ["reject", "exclude", "irrelevant"]
    )

    if has_accept and not has_reject:
        return 1

    if has_reject and not has_accept:
        return 0

    return np.nan


# ------------------------------------------------------------
# E. THEME NORMALISATION
# ------------------------------------------------------------

def normalise_theme(x):

    k = norm_key(x)

    if not k or k in {
        "nan",
        "none",
        "na",
        "theme",
        "themes",
        "if accept identify theme",
    }:
        return np.nan

    # Exact canonical match
    if k in THEME_NORM_MAP:
        return THEME_NORM_MAP[k]

    # Small punctuation / spacing differences
    for canonical_key, canonical_theme in THEME_NORM_MAP.items():

        if k == canonical_key:
            return canonical_theme

    return np.nan


# ------------------------------------------------------------
# F. TEXT CONSTRUCTION
# ------------------------------------------------------------

def build_text(title, abstract):

    title = clean_text(title)
    abstract = clean_text(abstract)

    if title and abstract:
        return f"Title: {title}. Abstract: {abstract}"

    return title or abstract


# ============================================================
# G. LOAD HUMAN-LABELLED TRAINING DATA
# ============================================================

def load_training(path):

    # IMPORTANT:
    # Your uploaded workbook uses the "Shreya" worksheet.
    raw = read_excel_smart(
        path,
        preferred_sheets=["Shreya"],
        expected_headers=[
            "Coder name",
            "Article ID",
            "Paper title",
            "Abstracts",
            "Accept/Reject",
            "If Accept, identify theme",
        ],
    )

    # Resolve columns robustly
    id_col = find_column(
        raw,
        [
            "Article ID",
            "Article_ID",
            "Paper ID",
            "ID",
        ]
    )

    title_col = find_column(
        raw,
        [
            "Paper title",
            "Paper_title",
            "Article Title",
            "Article_Title",
            "Title",
        ]
    )

    abstract_col = find_column(
        raw,
        [
            "Abstracts",
            "Abstract",
            "Paper Abstract",
        ]
    )

    decision_col = find_column(
        raw,
        [
            "Accept/Reject",
            "Accept Reject",
            "Decision",
            "Screening Decision",
        ]
    )

    theme_col = find_column(
        raw,
        [
            "If Accept, identify theme",
            "If Accept identify theme",
            "Theme",
            "Theme Label",
        ]
    )

    print("Resolved TRAINING columns:")
    print("  ID       :", id_col)
    print("  Title    :", title_col)
    print("  Abstract :", abstract_col)
    print("  Decision :", decision_col)
    print("  Theme    :", theme_col)

    df = pd.DataFrame({
        "paper_id": raw[id_col],
        "title": raw[title_col],
        "abstract": raw[abstract_col],
        "decision": raw[decision_col],
        "theme_raw": raw[theme_col],
    })

    # Clean text columns
    for c in [
        "paper_id",
        "title",
        "abstract",
        "decision",
        "theme_raw",
    ]:
        df[c] = df[c].apply(clean_text)

    # Build model text
    df["text"] = [
        build_text(t, a)
        for t, a in zip(
            df["title"],
            df["abstract"]
        )
    ]

    # Binary labels
    df["accept_label"] = (
        df["decision"]
        .apply(decision_to_binary)
    )

    # Theme labels
    df["theme"] = (
        df["theme_raw"]
        .apply(normalise_theme)
    )

    # --------------------------------------------------------
    # Your workbook contains:
    #
    # 1,521 Reject
    #   199 Accept
    #    15 Duplicate
    # ----------------
    # 1,735 raw rows
    #
    # Duplicate rows are excluded here -> 1,720 final rows
    # --------------------------------------------------------

    invalid_count = (
        df["accept_label"]
        .isna()
        .sum()
    )

    print(
        "\nRows excluded because they are "
        f"Duplicate / invalid decision labels: {invalid_count}"
    )

    df = (
        df[
            df["accept_label"].notna()
        ]
        .copy()
    )

    df["accept_label"] = (
        df["accept_label"]
        .astype(int)
    )

    # Remove genuinely empty records
    df = (
        df[
            df["text"].str.len() > 20
        ]
        .copy()
    )

    # Reject records have no valid Stage-2 theme
    df.loc[
        df["accept_label"].eq(0),
        "theme"
    ] = np.nan

    # Do NOT accidentally collapse different IDs unless
    # the complete modelling content + labels are identical.
    df = (
        df.drop_duplicates(
            subset=[
                "text",
                "accept_label",
                "theme",
            ]
        )
        .reset_index(drop=True)
    )

    df["norm_id"] = (
        df["paper_id"]
        .apply(norm_id)
    )

    return df


# ============================================================
# H. LOAD 10,176-RECORD MASTER CORPUS
# ============================================================

def load_master(path):

    raw = read_excel_smart(
        path,
        preferred_sheets=[
            "Formatted_ready to use Abstract"
        ],
        expected_headers=[
            "ID_OLD",
            "ID_New",
            "Article Title",
            "Abstract",
        ],
    )

    old_id_col = find_column(
        raw,
        ["ID_OLD", "ID OLD", "Old ID"]
    )

    new_id_col = find_column(
        raw,
        ["ID_New", "ID NEW", "New ID"]
    )

    title_col = find_column(
        raw,
        [
            "Article Title",
            "Article_Title",
            "Paper title",
            "Title",
        ]
    )

    abstract_col = find_column(
        raw,
        [
            "Abstract",
            "Abstracts",
        ]
    )

    df = raw.copy()

    # Preserve original ID columns expected later in the notebook
    if "ID_OLD" not in df.columns:
        df["ID_OLD"] = df[old_id_col]

    if "ID_New" not in df.columns:
        df["ID_New"] = df[new_id_col]

    old_ids = (
        df[old_id_col]
        .apply(clean_text)
    )

    new_ids = (
        df[new_id_col]
        .apply(clean_text)
    )

    # Prefer ID_New if present, otherwise ID_OLD
    df["paper_id"] = np.where(
        new_ids.str.len() > 0,
        new_ids,
        old_ids
    )

    df["title"] = (
        df[title_col]
        .apply(clean_text)
    )

    df["abstract"] = (
        df[abstract_col]
        .apply(clean_text)
    )

    df["text"] = [
        build_text(t, a)
        for t, a in zip(
            df["title"],
            df["abstract"]
        )
    ]

    df["norm_id"] = (
        df["paper_id"]
        .apply(norm_id)
    )

    return df.reset_index(drop=True)


# ============================================================
# I. LOAD 500-RECORD EXTERNAL AUDIT
# ============================================================

def load_audit(path):

    raw = read_excel_smart(
        path,
        preferred_sheets=["Sheet1"],
        expected_headers=[
            "ID_OLD",
            "ID_New",
            "Article_Title",
            "Abstract",
            "pred_decision",
            "pred_theme",
        ],
    )

    old_id_col = find_column(
        raw,
        ["ID_OLD", "ID OLD", "Old ID"]
    )

    new_id_col = find_column(
        raw,
        ["ID_New", "ID NEW", "New ID"]
    )

    title_col = find_column(
        raw,
        [
            "Article_Title",
            "Article Title",
            "Paper title",
            "Title",
        ]
    )

    abstract_col = find_column(
        raw,
        [
            "Abstract",
            "Abstracts",
        ]
    )

    df = raw.copy()

    old_ids = (
        df[old_id_col]
        .apply(clean_text)
    )

    new_ids = (
        df[new_id_col]
        .apply(clean_text)
    )

    df["paper_id"] = np.where(
        new_ids.str.len() > 0,
        new_ids,
        old_ids
    )

    df["title"] = (
        df[title_col]
        .apply(clean_text)
    )

    df["abstract"] = (
        df[abstract_col]
        .apply(clean_text)
    )

    df["text"] = [
        build_text(t, a)
        for t, a in zip(
            df["title"],
            df["abstract"]
        )
    ]

    df["norm_id"] = (
        df["paper_id"]
        .apply(norm_id)
    )

    # --------------------------------------------------------
    # Audit decision
    # --------------------------------------------------------

    decision_col = find_column(
        df,
        [
            AUDIT_DECISION_COL,
            "pred_decision",
        ],
        required=False
    )

    if decision_col is not None:

        df["audit_label"] = (
            df[decision_col]
            .apply(decision_to_binary)
        )

    else:

        df["audit_label"] = np.nan

    # --------------------------------------------------------
    # Audit theme
    # --------------------------------------------------------

    theme_col = find_column(
        df,
        [
            AUDIT_THEME_COL,
            "pred_theme",
        ],
        required=False
    )

    if theme_col is not None:

        df["audit_theme"] = (
            df[theme_col]
            .apply(normalise_theme)
        )

    else:

        df["audit_theme"] = np.nan

    return df.reset_index(drop=True)


# ============================================================
# J. LOAD ALL THREE FINAL DATASETS
# ============================================================

train_df = load_training(
    FILES["train"]
)

master_df = load_master(
    FILES["master"]
)

audit_df = load_audit(
    FILES["audit"]
)


# ============================================================
# K. CREATE THEME IDS
# ============================================================

theme_names = (
    CANONICAL_THEMES.copy()
)

theme_to_id = {
    theme: i
    for i, theme
    in enumerate(theme_names)
}

id_to_theme = {
    i: theme
    for theme, i
    in theme_to_id.items()
}

train_df["theme_id"] = (
    train_df["theme"]
    .map(theme_to_id)
)

audit_df["audit_theme_id"] = (
    audit_df["audit_theme"]
    .map(theme_to_id)
)


# ============================================================
# L. TRAIN / AUDIT / MASTER OVERLAP CHECK
# ============================================================

train_ids = set(
    train_df["norm_id"]
)

audit_ids = set(
    audit_df["norm_id"]
)

master_ids = set(
    master_df["norm_id"]
)


unseen_mask = (
    ~master_df["norm_id"]
    .isin(
        train_ids | audit_ids
    )
)

unseen_master = (
    master_df.loc[
        unseen_mask
    ]
    .copy()
    .reset_index(drop=True)
)


# ============================================================
# M. FINAL DATA CHECKS
# ============================================================

print("\n")
print("=" * 65)
print("FINAL DATA CHECK")
print("=" * 65)

print(
    "Training rows:",
    len(train_df)
)

print(
    "Reject / Accept:",
    (train_df["accept_label"] == 0).sum(),
    "/",
    (train_df["accept_label"] == 1).sum()
)

print(
    "Accepted with valid 12-theme label:",
    train_df["theme_id"].notna().sum()
)

print(
    "Master rows:",
    len(master_df)
)

print(
    "Audit rows:",
    len(audit_df)
)

print(
    "Training ∩ Audit IDs:",
    len(
        train_ids & audit_ids
    )
)

print(
    "Training IDs found in master:",
    master_df["norm_id"]
    .isin(train_ids)
    .sum()
)

print(
    "Audit IDs found in master:",
    master_df["norm_id"]
    .isin(audit_ids)
    .sum()
)

print(
    "Genuinely unseen master-demo pool:",
    len(unseen_master)
)


# ============================================================
# N. STRICT ASSERTIONS
# ============================================================

assert len(train_df) == 1720, (
    f"Expected 1,720 cleaned training rows, "
    f"got {len(train_df)}"
)

assert int(
    train_df["accept_label"].sum()
) == 199, (
    "Expected 199 Accept rows."
)

assert int(
    train_df["theme_id"]
    .notna()
    .sum()
) == 198, (
    "Expected 198 valid "
    "theme-labelled Accept rows."
)

assert len(master_df) == 10176, (
    f"Expected 10,176 master rows, "
    f"got {len(master_df)}"
)

assert len(audit_df) == 500, (
    f"Expected 500 audit rows, "
    f"got {len(audit_df)}"
)

assert len(
    train_ids & audit_ids
) == 0, (
    "Audit overlaps with "
    "labelled training data."
)

assert len(unseen_master) == 7956, (
    "Expected 7,956 unseen master rows "
    "after excluding 1,720 training + "
    f"500 audit, got {len(unseen_master)}"
)


# ============================================================
# O. THEME DISTRIBUTION
# ============================================================

print("\nTraining theme counts:")

display(
    train_df.loc[
        train_df["accept_label"].eq(1),
        "theme"
    ]
    .value_counts()
    .rename("count")
    .to_frame()
)


# ============================================================
# P. AUDIT DISTRIBUTION
# ============================================================

print(
    "\nAudit legacy-column distribution:"
)

display(
    audit_df[
        "audit_label"
    ]
    .value_counts(
        dropna=False
    )
    .rename(
        index={
            0: "Reject",
            1: "Accept"
        }
    )
    .to_frame("count")
)


# ============================================================
# Q. AUDIT MODE
# ============================================================

if AUDIT_LABELS_VERIFIED:

    assert (
        audit_df[
            "audit_label"
        ]
        .notna()
        .all()
    ), (
        "Audit contains missing "
        "decision labels."
    )

    print(
        "\nAUDIT MODE: VERIFIED HUMAN LABELS "
        "— external metrics will be computed."
    )

else:

    print(
        "\nAUDIT MODE: NOT VERIFIED "
        "— external metrics will be marked unavailable."
    )

    print(
        "The demo will follow the dissertation "
        "deployment policy: "
        "BGE+TF-IDF primary, "
        "FuseNet V2 challenger."
    )

Loaded: /kaggle/input/datasets/hrithikmajumdar/climate-text-dataset/Human labelled_DTU.xlsx
  sheet      = 'Shreya'
  header row = 1
  rows       = 1,735
  columns    = ['Coder name', 'Article ID', 'Paper_Author/s', 'Paper title', 'Year of publication', 'DOI', 'URL', 'Abstracts', 'Accept/Reject', 'If Accept, identify theme']

Resolved TRAINING columns:
  ID       : Article ID
  Title    : Paper title
  Abstract : Abstracts
  Decision : Accept/Reject
  Theme    : If Accept, identify theme

Rows excluded because they are Duplicate / invalid decision labels: 15
Loaded: /kaggle/input/datasets/hrithikmajumdar/climate-text-dataset/Master file_10k papers.xlsx
  sheet      = 'Formatted_ready to use Abstract'
  header row = 0
  rows       = 10,176
  columns    = ['ID_OLD', 'ID_New', 'Authors', 'Article Title', 'Publication Year', 'DOI', 'DOI Link', 'Abstract']

Loaded: /kaggle/input/datasets/hrithikmajumdar/climate-text-dataset/Test_sampled_data.xlsx
  sheet      = 'Sheet1'
  header row = 0
  r

,count
theme,
Climate change mitigation options & Well-being,84
"Empowering People & Governance, Policy",31
Implemented mitigation Options and Well-being,21
"Well-being, Community & Societal Systems Thinking",11
SDGs and Well-being,10
"Equity, Social Justice, Just Transition",8
Better Well-being Metric,7
Ecosystem & Non-human Well-being,6
Integrated Responses and Well-being,6



Audit legacy-column distribution:


,count
audit_label,
Reject,450
Accept,50



AUDIT MODE: NOT VERIFIED — external metrics will be marked unavailable.
The demo will follow the dissertation deployment policy: BGE+TF-IDF primary, FuseNet V2 challenger.


## Why the demonstration examples are genuinely unseen

The master workbook contains the same 1,720 records used for labelled development and the 500 audit records.  
The notebook therefore constructs a **strict demonstration pool** by excluding both sets of identifiers:

**10,176 master − 1,720 training − 500 audit = 7,956 unseen master records.**

Only these 7,956 records are eligible for the Gradio **“Load Unseen Master Record”** button.

In [6]:
# ============================================================
# 4. EMBEDDING CACHE: BGE + SPECTER
# ============================================================
def cache_key(name, texts):
    sig = f"{name}|n={len(texts)}|" + "|".join(texts[:2]) + "|" + "|".join(texts[-2:])
    return hashlib.md5(sig.encode("utf-8", errors="ignore")).hexdigest()[:12]

def get_bge_embeddings(texts, split_name):
    cache = CACHE_DIR / f"{split_name}_bge_{cache_key(BGE_MODEL_NAME, texts)}.npy"
    if cache.exists():
        print("Loading cached BGE:", cache)
        return np.load(cache)

    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(BGE_MODEL_NAME, device=DEVICE)
    emb = model.encode(
        texts,
        batch_size=EMBED_BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype(np.float32)
    np.save(cache, emb)
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return emb

def get_specter_embeddings(texts, split_name):
    cache = CACHE_DIR / f"{split_name}_specter_{cache_key(SPECTER_MODEL_NAME, texts)}.npy"
    if cache.exists():
        print("Loading cached SPECTER:", cache)
        return np.load(cache)

    from transformers import AutoTokenizer, AutoModel
    tok = AutoTokenizer.from_pretrained(SPECTER_MODEL_NAME)
    model = AutoModel.from_pretrained(SPECTER_MODEL_NAME).to(DEVICE).eval()

    vectors = []
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), EMBED_BATCH_SIZE), desc=f"SPECTER {split_name}"):
            batch = texts[i:i + EMBED_BATCH_SIZE]
            enc = tok(
                batch, padding=True, truncation=True, max_length=512,
                return_tensors="pt"
            )
            enc = {k:v.to(DEVICE) for k,v in enc.items()}
            out = model(**enc)
            pooled = out.last_hidden_state[:, 0, :]
            pooled = F.normalize(pooled, p=2, dim=1)
            vectors.append(pooled.cpu().numpy().astype(np.float32))

    emb = np.vstack(vectors)
    np.save(cache, emb)
    del model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return emb

train_texts = train_df["text"].tolist()
audit_texts = audit_df["text"].tolist()
master_texts = master_df["text"].tolist()

# BGE is required by BOTH candidate models.
BGE_TRAIN = get_bge_embeddings(train_texts, "train")
BGE_AUDIT = get_bge_embeddings(audit_texts, "audit")
BGE_MASTER = get_bge_embeddings(master_texts, "master")

# SPECTER is required only by FuseNet V2.
SPECTER_TRAIN = get_specter_embeddings(train_texts, "train")
SPECTER_AUDIT = get_specter_embeddings(audit_texts, "audit")
SPECTER_MASTER = get_specter_embeddings(master_texts, "master")

print("BGE:", BGE_TRAIN.shape, BGE_AUDIT.shape, BGE_MASTER.shape)
print("SPECTER:", SPECTER_TRAIN.shape, SPECTER_AUDIT.shape, SPECTER_MASTER.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

SPECTER audit:   0%|          | 0/16 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SPECTER master:   0%|          | 0/318 [00:00<?, ?it/s]

BGE: (1720, 384) (500, 384) (10176, 384)
SPECTER: (1720, 768) (500, 768) (10176, 768)


In [7]:
# ============================================================
# 5. SHARED METRICS AND THRESHOLD HELPERS
# ============================================================
def binary_metrics(y_true, probs, threshold):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=float)
    pred = (probs >= threshold).astype(int)
    ap = average_precision_score(y_true, probs)
    macro = f1_score(y_true, pred, average="macro", zero_division=0)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, labels=[0,1], zero_division=0
    )
    cm = confusion_matrix(y_true, pred, labels=[0,1])
    return {
        "AP": float(ap),
        "binary_macro_f1": float(macro),
        "accept_precision": float(precision[1]),
        "accept_recall": float(recall[1]),
        "accept_f1": float(f1[1]),
        "pred_accept_rate": float(pred.mean()),
        "tn": int(cm[0,0]), "fp": int(cm[0,1]),
        "fn": int(cm[1,0]), "tp": int(cm[1,1]),
    }

def tune_threshold(y_true, probs, target_accept_rate=None):
    y_true = np.asarray(y_true).astype(int)
    probs = np.asarray(probs, dtype=float)
    if target_accept_rate is None:
        target_accept_rate = float(y_true.mean())

    best = None
    for thr in np.linspace(0.02, 0.98, 193):
        m = binary_metrics(y_true, probs, thr)
        rate_penalty = abs(m["pred_accept_rate"] - target_accept_rate) / max(target_accept_rate, 1e-6)
        score = (
            0.55*m["binary_macro_f1"]
            + 0.30*m["accept_f1"]
            + 0.15*m["AP"]
            - 0.08*rate_penalty
        )
        if best is None or score > best["score"]:
            best = {"threshold": float(thr), "score": float(score), **m}
    return best

def theme_metrics(y_true, probs, labels=None):
    y_true = np.asarray(y_true).astype(int)
    pred = np.asarray(probs).argmax(axis=1)
    if labels is None:
        labels = list(range(len(theme_names)))
    return {
        "theme_macro_f1": float(f1_score(y_true, pred, labels=labels, average="macro", zero_division=0)),
        "theme_weighted_f1": float(f1_score(y_true, pred, labels=labels, average="weighted", zero_division=0)),
        "theme_accuracy": float(accuracy_score(y_true, pred)),
        "theme_coverage": int(len(np.unique(pred))),
    }

def prior_correct(probs, class_counts, alpha=0.0):
    probs = np.asarray(probs, dtype=float)
    if alpha <= 0:
        return probs / np.maximum(probs.sum(axis=1, keepdims=True), 1e-12)
    counts = np.asarray(class_counts, dtype=float) + 1.0
    prior = counts / counts.sum()
    p = probs / np.power(prior[None,:], alpha)
    p = np.clip(p, 1e-12, None)
    return p / p.sum(axis=1, keepdims=True)

TRAIN_ACCEPT_RATE = float(train_df["accept_label"].mean())
print("Training Accept prevalence:", TRAIN_ACCEPT_RATE)

Training Accept prevalence: 0.11569767441860465


In [8]:
# ============================================================
# 6. MODEL A — BGE + TF-IDF CLASSICAL HIERARCHICAL PIPELINE
# ============================================================
def fit_tfidf_vectorizers(texts):
    word_vec = TfidfVectorizer(
        analyzer="word", ngram_range=(1,2), min_df=2, max_df=0.95,
        max_features=WORD_MAX_FEATURES, sublinear_tf=True, strip_accents="unicode"
    )
    char_vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3,5), min_df=2,
        max_features=CHAR_MAX_FEATURES, sublinear_tf=True
    )
    Xw = word_vec.fit_transform(texts)
    Xc = char_vec.fit_transform(texts)
    return word_vec, char_vec, sp.hstack([Xw, Xc]).tocsr()

def transform_tfidf(word_vec, char_vec, texts):
    return sp.hstack([
        word_vec.transform(texts),
        char_vec.transform(texts)
    ]).tocsr()

def combine_classical(X_tfidf, X_bge):
    return sp.hstack([
        X_tfidf.tocsr(),
        sp.csr_matrix(np.asarray(X_bge, dtype=np.float32) * EMBEDDING_WEIGHT)
    ]).tocsr()

def make_calibrated_svc(C=0.75):
    base = LinearSVC(C=C, class_weight="balanced", max_iter=6000, random_state=SEED)
    try:
        return CalibratedClassifierCV(estimator=base, method="sigmoid", cv=3)
    except TypeError:
        return CalibratedClassifierCV(base_estimator=base, method="sigmoid", cv=3)

def safe_binary_prob(model, X):
    if hasattr(model, "predict_proba"):
        p = model.predict_proba(X)
        classes = list(getattr(model, "classes_", [0,1]))
        return p[:, classes.index(1)]
    return expit(model.decision_function(X))

def train_stage1_ensemble(X_combined, y, X_tfidf):
    models = []
    lr = LogisticRegression(
        C=1.3, max_iter=4000, class_weight="balanced",
        solver="liblinear", random_state=SEED
    ).fit(X_combined, y)
    models.append(("LR", lr, 0.34, "combined"))

    svc = make_calibrated_svc(0.75).fit(X_combined, y)
    models.append(("CalibratedLinearSVC", svc, 0.34, "combined"))

    sgd = SGDClassifier(
        loss="log_loss", alpha=1e-5, penalty="elasticnet", l1_ratio=0.10,
        class_weight="balanced", max_iter=2000, tol=1e-4, random_state=SEED
    ).fit(X_combined, y)
    models.append(("SGD", sgd, 0.20, "combined"))

    nbm = ComplementNB(alpha=0.15).fit(X_tfidf, y)
    models.append(("ComplementNB", nbm, 0.12, "tfidf"))
    return models

def predict_stage1_ensemble(models, X_combined, X_tfidf):
    total = np.zeros(X_combined.shape[0], dtype=float)
    weight_sum = 0.0
    for _, model, w, kind in models:
        X = X_tfidf if kind == "tfidf" else X_combined
        total += w * safe_binary_prob(model, X)
        weight_sum += w
    return total / weight_sum

def align_multiclass(model, X, n_classes):
    if hasattr(model, "predict_proba"):
        raw = model.predict_proba(X)
        classes = model.classes_
    else:
        scores = np.asarray(model.decision_function(X), dtype=float)
        if scores.ndim == 1:
            scores = np.c_[-scores, scores]
        raw = softmax(scores, axis=1)
        classes = model.classes_
    out = np.zeros((X.shape[0], n_classes), dtype=float)
    for j, c in enumerate(classes):
        out[:, int(c)] = raw[:, j]
    bad = out.sum(axis=1) == 0
    out[bad] = 1.0 / n_classes
    return out / out.sum(axis=1, keepdims=True)

def train_theme_ensemble(X_combined, y, X_tfidf):
    models = []
    lr = LogisticRegression(
        C=1.7, max_iter=5000, class_weight="balanced",
        solver="lbfgs", random_state=SEED
    ).fit(X_combined, y)
    models.append(("ThemeLR", lr, 0.45, "combined"))

    svc = LinearSVC(
        C=0.75, class_weight="balanced", max_iter=6000, random_state=SEED
    ).fit(X_combined, y)
    models.append(("ThemeSVC", svc, 0.30, "combined"))

    sgd = SGDClassifier(
        loss="log_loss", alpha=1e-4, penalty="l2", class_weight="balanced",
        max_iter=3000, tol=1e-4, random_state=SEED
    ).fit(X_combined, y)
    models.append(("ThemeSGD", sgd, 0.15, "combined"))

    nbm = ComplementNB(alpha=0.35).fit(X_tfidf, y)
    models.append(("ThemeNB", nbm, 0.10, "tfidf"))
    return models

def predict_theme_ensemble(models, X_combined, X_tfidf, n_classes):
    total = np.zeros((X_combined.shape[0], n_classes), dtype=float)
    weight_sum = 0.0
    for _, model, w, kind in models:
        X = X_tfidf if kind == "tfidf" else X_combined
        total += w * align_multiclass(model, X, n_classes)
        weight_sum += w
    p = total / weight_sum
    return p / p.sum(axis=1, keepdims=True)

def prototype_probs(train_repr, y_train, eval_repr, n_classes, temperature=0.08):
    A = normalize(np.asarray(train_repr, dtype=float), norm="l2")
    B = normalize(np.asarray(eval_repr, dtype=float), norm="l2")
    centroids = np.zeros((n_classes, A.shape[1]))
    for c in range(n_classes):
        centroids[c] = A[y_train == c].mean(axis=0)
    centroids = normalize(centroids, norm="l2")
    return softmax((B @ centroids.T) / temperature, axis=1)

print("Classical model helpers ready.")

Classical model helpers ready.


In [9]:
# ============================================================
# 7. BGE + TF-IDF — FOLD-SAFE INTERNAL CV
# ============================================================
y_bin = train_df["accept_label"].values.astype(int)

oof_raw = np.zeros(len(train_df), dtype=float)
stage1_fold_rows = []
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr, va) in enumerate(skf.split(np.zeros(len(y_bin)), y_bin), 1):
    print(f"Classical Stage 1 fold {fold}/{N_FOLDS}")
    wv, cv, Xtr_tf = fit_tfidf_vectorizers([train_texts[i] for i in tr])
    Xva_tf = transform_tfidf(wv, cv, [train_texts[i] for i in va])
    Xtr = combine_classical(Xtr_tf, BGE_TRAIN[tr])
    Xva = combine_classical(Xva_tf, BGE_TRAIN[va])

    models = train_stage1_ensemble(Xtr, y_bin[tr], Xtr_tf)
    p = predict_stage1_ensemble(models, Xva, Xva_tf)
    oof_raw[va] = p

    temp_thr = tune_threshold(y_bin[va], p, float(y_bin[tr].mean()))
    stage1_fold_rows.append({"fold": fold, **temp_thr})
    del Xtr, Xva, Xtr_tf, Xva_tf, models
    gc.collect()

classical_calibrator = IsotonicRegression(out_of_bounds="clip")
classical_calibrator.fit(oof_raw, y_bin)
classical_oof_prob = classical_calibrator.transform(oof_raw)
CLASSICAL_THRESHOLD = tune_threshold(y_bin, classical_oof_prob, TRAIN_ACCEPT_RATE)["threshold"]
classical_stage1_internal = binary_metrics(y_bin, classical_oof_prob, CLASSICAL_THRESHOLD)

# Stage 2: accepted-only CV. Smallest theme has 3 examples, so at most 3 stratified folds.
accepted_idx = np.where(train_df["theme_id"].notna().values)[0]
y_theme = train_df.loc[accepted_idx, "theme_id"].astype(int).values
theme_counts = np.bincount(y_theme, minlength=len(theme_names))
theme_folds = min(N_FOLDS, int(theme_counts[theme_counts > 0].min()))

theme_oof = np.zeros((len(accepted_idx), len(theme_names)), dtype=float)
tskf = StratifiedKFold(n_splits=theme_folds, shuffle=True, random_state=SEED)

for fold, (tr_rel, va_rel) in enumerate(tskf.split(np.zeros(len(y_theme)), y_theme), 1):
    print(f"Classical Stage 2 fold {fold}/{theme_folds}")
    tr_abs, va_abs = accepted_idx[tr_rel], accepted_idx[va_rel]
    wv, cv, Xtr_tf = fit_tfidf_vectorizers([train_texts[i] for i in tr_abs])
    Xva_tf = transform_tfidf(wv, cv, [train_texts[i] for i in va_abs])
    Xtr = combine_classical(Xtr_tf, BGE_TRAIN[tr_abs])
    Xva = combine_classical(Xva_tf, BGE_TRAIN[va_abs])

    models = train_theme_ensemble(Xtr, y_theme[tr_rel], Xtr_tf)
    p_cls = predict_theme_ensemble(models, Xva, Xva_tf, len(theme_names))
    p_proto = prototype_probs(
        BGE_TRAIN[tr_abs], y_theme[tr_rel], BGE_TRAIN[va_abs],
        len(theme_names), temperature=0.08
    )
    theme_oof[va_rel] = 0.72*p_cls + 0.28*p_proto
    del Xtr, Xva, Xtr_tf, Xva_tf, models
    gc.collect()

alpha_rows = []
for alpha in [0.0, 0.10, 0.20, 0.35, 0.50, 0.75, 1.0]:
    p = prior_correct(theme_oof, theme_counts, alpha)
    alpha_rows.append({"alpha": alpha, **theme_metrics(y_theme, p)})
alpha_df = pd.DataFrame(alpha_rows).sort_values(
    ["theme_macro_f1","theme_weighted_f1"], ascending=False
).reset_index(drop=True)
CLASSICAL_THEME_ALPHA = float(alpha_df.iloc[0]["alpha"])
classical_theme_internal = theme_metrics(
    y_theme, prior_correct(theme_oof, theme_counts, CLASSICAL_THEME_ALPHA)
)

print("\nBGE + TF-IDF internal results")
display(pd.DataFrame([{**classical_stage1_internal, **classical_theme_internal,
                       "threshold": CLASSICAL_THRESHOLD,
                       "theme_prior_alpha": CLASSICAL_THEME_ALPHA}]).T)

Classical Stage 1 fold 1/5
Classical Stage 1 fold 2/5
Classical Stage 1 fold 3/5
Classical Stage 1 fold 4/5
Classical Stage 1 fold 5/5
Classical Stage 2 fold 1/3
Classical Stage 2 fold 2/3
Classical Stage 2 fold 3/3

BGE + TF-IDF internal results


,0
AP,0.427884
binary_macro_f1,0.713215
accept_precision,0.505319
accept_recall,0.477387
accept_f1,0.490956
pred_accept_rate,0.109302
tn,1428.000000
fp,93.000000
fn,104.000000
tp,95.000000


In [10]:
# ============================================================
# 8. FIT FINAL BGE + TF-IDF MODEL ON ALL LABELLED DATA
# ============================================================
# Stage 1 full fit
CL_WV1, CL_CV1, CL_XTF1 = fit_tfidf_vectorizers(train_texts)
CL_X1 = combine_classical(CL_XTF1, BGE_TRAIN)
CL_STAGE1_MODELS = train_stage1_ensemble(CL_X1, y_bin, CL_XTF1)

# Stage 2 full fit
accepted_texts = [train_texts[i] for i in accepted_idx]
CL_WV2, CL_CV2, CL_XTF2 = fit_tfidf_vectorizers(accepted_texts)
CL_X2 = combine_classical(CL_XTF2, BGE_TRAIN[accepted_idx])
CL_STAGE2_MODELS = train_theme_ensemble(CL_X2, y_theme, CL_XTF2)
CL_THEME_BGE = BGE_TRAIN[accepted_idx].copy()
CL_THEME_Y = y_theme.copy()
CL_THEME_COUNTS = theme_counts.copy()

def classical_predict_batch(texts, bge_embeddings):
    # Stage 1
    xtf1 = transform_tfidf(CL_WV1, CL_CV1, texts)
    x1 = combine_classical(xtf1, bge_embeddings)
    raw = predict_stage1_ensemble(CL_STAGE1_MODELS, x1, xtf1)
    p_accept = classical_calibrator.transform(raw)
    decisions = (p_accept >= CLASSICAL_THRESHOLD).astype(int)

    # Stage 2 probabilities are computed for all rows, but exposed only if accepted.
    xtf2 = transform_tfidf(CL_WV2, CL_CV2, texts)
    x2 = combine_classical(xtf2, bge_embeddings)
    p_cls = predict_theme_ensemble(CL_STAGE2_MODELS, x2, xtf2, len(theme_names))
    p_proto = prototype_probs(
        CL_THEME_BGE, CL_THEME_Y, bge_embeddings,
        len(theme_names), temperature=0.08
    )
    p_theme = prior_correct(
        0.72*p_cls + 0.28*p_proto,
        CL_THEME_COUNTS,
        CLASSICAL_THEME_ALPHA
    )
    return p_accept, decisions, p_theme

print("Final BGE + TF-IDF models fitted.")

Final BGE + TF-IDF models fitted.


In [11]:
# ============================================================
# 9. CLIMATEWELL-FUSENET V2 PREPROCESSING + ARCHITECTURE
# ============================================================
def fit_tfidf_svd(train_texts, eval_texts=None):
    word_vec = TfidfVectorizer(
        analyzer="word", ngram_range=(1,2), min_df=2, max_df=0.95,
        max_features=25_000, sublinear_tf=True, strip_accents="unicode"
    )
    char_vec = TfidfVectorizer(
        analyzer="char_wb", ngram_range=(3,5), min_df=2, max_df=0.95,
        max_features=25_000, sublinear_tf=True, strip_accents="unicode"
    )
    Xs = sp.hstack([
        word_vec.fit_transform(train_texts),
        char_vec.fit_transform(train_texts)
    ]).tocsr()

    ncomp = min(SVD_DIM, Xs.shape[0]-1, Xs.shape[1]-1)
    svd = TruncatedSVD(n_components=max(2, ncomp), random_state=SEED)
    Xtr = svd.fit_transform(Xs).astype(np.float32)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr).astype(np.float32)

    def transform(texts):
        if texts is None:
            return None
        X = sp.hstack([
            word_vec.transform(texts), char_vec.transform(texts)
        ]).tocsr()
        return scaler.transform(svd.transform(X)).astype(np.float32)

    return {
        "word_vec": word_vec, "char_vec": char_vec,
        "svd": svd, "scaler": scaler,
        "train": Xtr, "eval": transform(eval_texts)
    }

def fit_dense_scaler(train_arr, eval_arr=None):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(train_arr).astype(np.float32)
    Xev = scaler.transform(eval_arr).astype(np.float32) if eval_arr is not None else None
    return Xtr, Xev, scaler

class ClimateDataset(Dataset):
    def __init__(self, bge, specter, tfidf, y_bin, y_theme):
        self.bge = torch.tensor(bge, dtype=torch.float32)
        self.specter = torch.tensor(specter, dtype=torch.float32)
        self.tfidf = torch.tensor(tfidf, dtype=torch.float32)
        self.y_bin = torch.tensor(y_bin, dtype=torch.float32)
        self.y_theme = torch.tensor(y_theme, dtype=torch.long)
    def __len__(self):
        return len(self.y_bin)
    def __getitem__(self, i):
        return {
            "bge": self.bge[i], "specter": self.specter[i], "tfidf": self.tfidf[i],
            "y_bin": self.y_bin[i], "y_theme": self.y_theme[i]
        }

class ClimateWellFuseNet(nn.Module):
    def __init__(self, bge_dim, specter_dim, tfidf_dim, n_themes):
        super().__init__()
        def projector(in_dim):
            return nn.Sequential(
                nn.Linear(in_dim, HIDDEN_DIM), nn.LayerNorm(HIDDEN_DIM),
                nn.GELU(), nn.Dropout(DROPOUT),
                nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.LayerNorm(HIDDEN_DIM), nn.GELU()
            )
        self.bge_proj = projector(bge_dim)
        self.specter_proj = projector(specter_dim)
        self.tfidf_proj = projector(tfidf_dim)

        self.gate_net = nn.Sequential(
            nn.Linear(HIDDEN_DIM*3, HIDDEN_DIM), nn.GELU(),
            nn.Dropout(DROPOUT), nn.Linear(HIDDEN_DIM, 3)
        )
        self.concat_context = nn.Sequential(
            nn.Linear(HIDDEN_DIM*3, HIDDEN_DIM), nn.LayerNorm(HIDDEN_DIM),
            nn.GELU(), nn.Dropout(DROPOUT)
        )
        self.shared = nn.Sequential(
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.LayerNorm(HIDDEN_DIM),
            nn.GELU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.LayerNorm(HIDDEN_DIM), nn.GELU()
        )
        self.binary_head = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(HIDDEN_DIM, HIDDEN_DIM//2),
            nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(HIDDEN_DIM//2,1)
        )
        self.theme_head = nn.Sequential(
            nn.Dropout(DROPOUT), nn.Linear(HIDDEN_DIM, HIDDEN_DIM//2),
            nn.GELU(), nn.Dropout(DROPOUT), nn.Linear(HIDDEN_DIM//2,n_themes)
        )
        self.theme_prototypes = nn.Parameter(torch.randn(n_themes, HIDDEN_DIM)*0.02)

    def _mask(self, n, device):
        mask = torch.ones((n,3), dtype=torch.bool, device=device)
        if self.training and BRANCH_DROPOUT_PROB > 0:
            keep = torch.rand((n,3), device=device) > BRANCH_DROPOUT_PROB
            mask = mask & keep
            empty = mask.sum(1).eq(0)
            mask[empty] = True
        return mask

    def forward(self, bge, specter, tfidf):
        pb, ps, pt = self.bge_proj(bge), self.specter_proj(specter), self.tfidf_proj(tfidf)
        mask = self._mask(len(pb), pb.device)
        mf = mask.float()

        gate_logits = self.gate_net(torch.cat([pb,ps,pt],1)).masked_fill(~mask, -1e9)
        gates = torch.softmax(gate_logits, 1)
        active = mf.sum(1,keepdim=True).clamp(min=1)
        floor = torch.where(active > 1, torch.full_like(active, GATE_FLOOR), torch.zeros_like(active))
        gates = gates*(1-floor*active) + mf*floor
        gates = gates/gates.sum(1,keepdim=True).clamp(min=1e-8)

        residual = self.concat_context(torch.cat([pb*mf[:,0:1], ps*mf[:,1:2], pt*mf[:,2:3]],1))
        fused = gates[:,0:1]*pb + gates[:,1:2]*ps + gates[:,2:3]*pt + residual
        z = self.shared(fused)

        binary_logits = self.binary_head(z).squeeze(1)
        theme_cls = self.theme_head(z)
        proto = (
            F.normalize(z,p=2,dim=1)
            @ F.normalize(self.theme_prototypes,p=2,dim=1).T
        ) / PROTO_TEMPERATURE
        theme_logits = theme_cls + PROTO_LOGIT_WEIGHT*proto
        entropy = -(gates*torch.log(gates.clamp(min=1e-8))).sum(1)
        return {
            "binary_logits": binary_logits,
            "theme_logits": theme_logits,
            "gates": gates,
            "gate_entropy": entropy
        }

print("ClimateWell-FuseNet V2 architecture ready.")

ClimateWell-FuseNet V2 architecture ready.


In [12]:
# ============================================================
# 10. FUSENET TRAINING FUNCTIONS
# ============================================================
y_theme_full = train_df["theme_id"].fillna(-1).astype(int).values

def focal_bce(logits, targets, pos_weight, gamma=BINARY_FOCAL_GAMMA):
    bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction="none")
    p = torch.sigmoid(logits)
    pt = torch.where(targets.eq(1), p, 1-p)
    return (((1-pt).clamp(min=1e-4)**gamma)*bce).mean()

def focal_ce(logits, targets, weight, gamma=THEME_FOCAL_GAMMA):
    ce = F.cross_entropy(
        logits, targets, weight=weight,
        label_smoothing=THEME_LABEL_SMOOTHING, reduction="none"
    )
    p = torch.softmax(logits,1).gather(1,targets[:,None]).squeeze(1).clamp(min=1e-4)
    return (((1-p)**gamma)*ce).mean()

def gate_reg(out):
    g = out["gates"]
    ent = out["gate_entropy"].mean()/math.log(3)
    target = torch.full((3,),1/3,device=g.device)
    balance = F.mse_loss(g.mean(0), target)
    return -GATE_ENTROPY_WEIGHT*ent + GATE_BALANCE_WEIGHT*balance

def make_train_loader(Xb,Xs,Xt,yb,yt,idx):
    ds = ClimateDataset(Xb[idx],Xs[idx],Xt[idx],yb[idx],yt[idx])
    yy = yb[idx]
    pos, neg = max(1,int(yy.sum())), max(1,int(len(yy)-yy.sum()))
    weights = np.where(yy==1,0.5/pos,0.5/neg).astype(np.float32)
    sampler = WeightedRandomSampler(torch.tensor(weights), len(weights), replacement=True)
    return DataLoader(ds,batch_size=BATCH_SIZE,sampler=sampler,num_workers=0)

def make_eval_loader(Xb,Xs,Xt,yb,yt,idx):
    ds = ClimateDataset(Xb[idx],Xs[idx],Xt[idx],yb[idx],yt[idx])
    return DataLoader(ds,batch_size=BATCH_SIZE*2,shuffle=False,num_workers=0)

def predict_fusenet_loader(model, loader):
    model.eval()
    bp, tp, gp = [], [], []
    with torch.no_grad():
        for b in loader:
            out = model(
                b["bge"].to(DEVICE), b["specter"].to(DEVICE), b["tfidf"].to(DEVICE)
            )
            bp.append(torch.sigmoid(out["binary_logits"]).cpu().numpy())
            tp.append(torch.softmax(out["theme_logits"],1).cpu().numpy())
            gp.append(out["gates"].cpu().numpy())
    return np.concatenate(bp), np.vstack(tp), np.vstack(gp)

def theme_weights_for_idx(y_theme, idx):
    valid = y_theme[idx][y_theme[idx] >= 0]
    counts = np.bincount(valid, minlength=len(theme_names)).astype(np.float32)
    w = np.sqrt(max(1, counts.sum())/np.maximum(counts,1))
    w = np.clip(w/w.mean(),0.5,4.0)
    return torch.tensor(w,dtype=torch.float32,device=DEVICE)

def train_fusenet_fold(Xb,Xs,Xt,yb,yt,tr,va):
    model = ClimateWellFuseNet(Xb.shape[1],Xs.shape[1],Xt.shape[1],len(theme_names)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=MAX_EPOCHS)

    tr_loader = make_train_loader(Xb,Xs,Xt,yb,yt,tr)
    va_loader = make_eval_loader(Xb,Xs,Xt,yb,yt,va)

    pos, neg = max(1,int(yb[tr].sum())), max(1,int(len(tr)-yb[tr].sum()))
    pos_weight = torch.tensor([math.sqrt(neg/pos)],dtype=torch.float32,device=DEVICE)
    th_w = theme_weights_for_idx(yt,tr)

    best_score, best_state, best_epoch, bad = -1e9, None, 1, 0
    target_rate = float(yb[tr].mean())

    for epoch in range(1,MAX_EPOCHS+1):
        model.train()
        for b in tr_loader:
            out = model(
                b["bge"].to(DEVICE), b["specter"].to(DEVICE), b["tfidf"].to(DEVICE)
            )
            ybb = b["y_bin"].to(DEVICE)
            ytt = b["y_theme"].to(DEVICE)

            loss_bin = focal_bce(out["binary_logits"],ybb,pos_weight)
            mask = ytt >= 0
            loss_theme = focal_ce(out["theme_logits"][mask],ytt[mask],th_w) if mask.any() else torch.tensor(0.,device=DEVICE)
            loss = loss_bin + THEME_LOSS_WEIGHT*loss_theme + gate_reg(out)

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()
        sched.step()

        bp,tp,_ = predict_fusenet_loader(model,va_loader)
        bm = tune_threshold(yb[va],bp,target_rate)
        valid = yt[va] >= 0
        tm = theme_metrics(yt[va][valid],tp[valid]) if valid.any() else {"theme_macro_f1":0,"theme_weighted_f1":0}
        score = (
            0.25*bm["AP"] + 0.20*bm["accept_f1"] + 0.15*bm["binary_macro_f1"]
            + 0.35*tm["theme_macro_f1"] + 0.05*tm["theme_weighted_f1"]
        )
        if score > best_score:
            best_score, best_epoch, bad = score, epoch, 0
            best_state = {k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break

    model.load_state_dict(best_state)
    bp,tp,gp = predict_fusenet_loader(model,va_loader)
    return model,bp,tp,gp,best_epoch,best_score

def train_fusenet_all(Xb,Xs,Xt,yb,yt,epochs):
    idx = np.arange(len(yb))
    model = ClimateWellFuseNet(Xb.shape[1],Xs.shape[1],Xt.shape[1],len(theme_names)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(1,epochs))
    loader = make_train_loader(Xb,Xs,Xt,yb,yt,idx)

    pos, neg = max(1,int(yb.sum())), max(1,int(len(yb)-yb.sum()))
    pos_weight = torch.tensor([math.sqrt(neg/pos)],dtype=torch.float32,device=DEVICE)
    th_w = theme_weights_for_idx(yt,idx)

    for epoch in tqdm(range(1,epochs+1),desc="Final FuseNet fit"):
        model.train()
        for b in loader:
            out = model(
                b["bge"].to(DEVICE), b["specter"].to(DEVICE), b["tfidf"].to(DEVICE)
            )
            ybb, ytt = b["y_bin"].to(DEVICE), b["y_theme"].to(DEVICE)
            lb = focal_bce(out["binary_logits"],ybb,pos_weight)
            mask = ytt >= 0
            lt = focal_ce(out["theme_logits"][mask],ytt[mask],th_w) if mask.any() else torch.tensor(0.,device=DEVICE)
            loss = lb + THEME_LOSS_WEIGHT*lt + gate_reg(out)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step()
        sched.step()
    return model

print("FuseNet training helpers ready.")

FuseNet training helpers ready.


In [15]:
# ============================================================
# 11. CLIMATEWELL-FUSENET V2 — STABLE 5-FOLD OOF EVALUATION
# ============================================================
#
# Fixes:
# 1. Stable cosine/prototype normalisation.
# 2. Prototype re-normalisation after optimiser updates.
# 3. Non-finite gradients trigger controlled early stopping
#    and restoration of the best FINITE checkpoint.
# 4. No fake NaN -> 0.5 probability replacement.
# 5. Keeps the dissertation LR = 1.2e-3.
# 6. Overrides train_fusenet_all as well, so Cell 12 will
#    inherit the same numerical protection.
# ============================================================

import gc
import math
import random
import copy

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import StratifiedKFold


# ============================================================
# 11A. REPRODUCIBILITY
# ============================================================

def reset_all_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


reset_all_seeds(SEED)


# ============================================================
# 11B. INITIAL FEATURE CHECK
# ============================================================

def report_finite(name, x):

    x = np.asarray(x)

    print(
        f"{name:<25}"
        f" shape={str(x.shape):<18}"
        f" NaN={np.isnan(x).sum():<7}"
        f" +Inf={np.isposinf(x).sum():<7}"
        f" -Inf={np.isneginf(x).sum():<7}"
    )


print("=" * 80)
print("PRE-FUSENET NUMERICAL CHECK")
print("=" * 80)

report_finite(
    "BGE_TRAIN",
    BGE_TRAIN
)

report_finite(
    "SPECTER_TRAIN",
    SPECTER_TRAIN
)


assert np.isfinite(BGE_TRAIN).all(), (
    "BGE_TRAIN contains NaN/Inf."
)

assert np.isfinite(SPECTER_TRAIN).all(), (
    "SPECTER_TRAIN contains NaN/Inf."
)


# ============================================================
# 11C. STABLE NORMALISATION
# ============================================================
#
# Default F.normalize uses eps=1e-12.
#
# That is harmless in the forward pass but can produce huge
# gradients when a vector norm gets extremely close to zero.
#
# 1e-5 only changes behaviour for an effectively zero vector.
# ============================================================

STABLE_NORM_EPS = 1e-5


def stable_l2_normalize(x, dim=1):

    norm = torch.linalg.vector_norm(
        x,
        ord=2,
        dim=dim,
        keepdim=True
    )

    norm = torch.clamp(
        norm,
        min=STABLE_NORM_EPS
    )

    return x / norm


# ============================================================
# 11D. PATCH FUSENET FORWARD PASS
# ============================================================
#
# Same architecture as before.
#
# Only numerical difference:
# cosine vectors use stable_l2_normalize().
# ============================================================

def stable_fusenet_forward(
    self,
    bge,
    specter,
    tfidf
):

    # --------------------------------------------------------
    # Branch projections
    # --------------------------------------------------------

    pb = self.bge_proj(bge)

    ps = self.specter_proj(
        specter
    )

    pt = self.tfidf_proj(
        tfidf
    )


    batch_size = pb.shape[0]


    # --------------------------------------------------------
    # Existing branch-dropout logic
    # --------------------------------------------------------

    if hasattr(
        self,
        "_active_mask"
    ):

        mask = self._active_mask(
            batch_size,
            pb.device
        )

    else:

        # Compatible with the simplified class used
        # in the defence notebook.

        mask = torch.ones(
            (
                batch_size,
                3
            ),
            dtype=torch.bool,
            device=pb.device
        )

        if (
            self.training
            and
            BRANCH_DROPOUT_PROB > 0
        ):

            keep = (
                torch.rand(
                    (
                        batch_size,
                        3
                    ),
                    device=pb.device
                )
                >
                BRANCH_DROPOUT_PROB
            )

            mask = (
                mask
                &
                keep
            )

            empty = (
                mask
                .sum(dim=1)
                .eq(0)
            )

            if empty.any():

                mask[
                    empty
                ] = True


    mask_f = mask.float()


    # --------------------------------------------------------
    # Gate
    # --------------------------------------------------------

    raw_gate_logits = (
        self.gate_net(
            torch.cat(
                [
                    pb,
                    ps,
                    pt
                ],
                dim=1
            )
        )
    )


    gate_logits = (
        raw_gate_logits
        .masked_fill(
            ~mask,
            -1e4
        )
    )


    gates = torch.softmax(
        gate_logits,
        dim=1
    )


    # --------------------------------------------------------
    # Gate floor
    # --------------------------------------------------------

    active_counts = (
        mask_f
        .sum(
            dim=1,
            keepdim=True
        )
        .clamp(
            min=1.0
        )
    )


    floor = torch.where(
        active_counts > 1,
        torch.full_like(
            active_counts,
            GATE_FLOOR
        ),
        torch.zeros_like(
            active_counts
        )
    )


    gates = (
        gates
        *
        (
            1.0
            -
            floor
            *
            active_counts
        )
        +
        mask_f
        *
        floor
    )


    gates = (
        gates
        /
        gates
        .sum(
            dim=1,
            keepdim=True
        )
        .clamp(
            min=1e-6
        )
    )


    # --------------------------------------------------------
    # Residual context
    # --------------------------------------------------------

    pb_m = (
        pb
        *
        mask_f[
            :,
            0:1
        ]
    )

    ps_m = (
        ps
        *
        mask_f[
            :,
            1:2
        ]
    )

    pt_m = (
        pt
        *
        mask_f[
            :,
            2:3
        ]
    )


    weighted_fused = (
        gates[:, 0:1] * pb
        +
        gates[:, 1:2] * ps
        +
        gates[:, 2:3] * pt
    )


    residual_context = (
        self.concat_context(
            torch.cat(
                [
                    pb_m,
                    ps_m,
                    pt_m
                ],
                dim=1
            )
        )
    )


    fused = (
        weighted_fused
        +
        residual_context
    )


    z = self.shared(
        fused
    )


    # --------------------------------------------------------
    # Stage 1 head
    # --------------------------------------------------------

    binary_logits = (
        self.binary_head(
            z
        )
        .squeeze(1)
    )


    # --------------------------------------------------------
    # Stage 2 classifier head
    # --------------------------------------------------------

    theme_cls_logits = (
        self.theme_head(
            z
        )
    )


    # ========================================================
    # IMPORTANT NUMERICAL FIX
    # ========================================================

    z_norm = (
        stable_l2_normalize(
            z,
            dim=1
        )
    )


    proto_norm = (
        stable_l2_normalize(
            self.theme_prototypes,
            dim=1
        )
    )


    proto_logits = (
        torch.matmul(
            z_norm,
            proto_norm.T
        )
        /
        PROTO_TEMPERATURE
    )


    theme_logits = (
        theme_cls_logits
        +
        PROTO_LOGIT_WEIGHT
        *
        proto_logits
    )


    gate_entropy = -(
        gates
        *
        torch.log(
            gates.clamp(
                min=1e-6
            )
        )
    ).sum(
        dim=1
    )


    return {
        "binary_logits":
            binary_logits,

        "theme_logits":
            theme_logits,

        "gates":
            gates,

        "gate_entropy":
            gate_entropy,

        "representation":
            z,
    }


# Replace the previous class method.
ClimateWellFuseNet.forward = (
    stable_fusenet_forward
)

print(
    "\n✓ Patched ClimateWellFuseNet.forward "
    "with stable prototype normalisation."
)


# ============================================================
# 11E. KEEP RAW PROTOTYPES NUMERICALLY WELL-CONDITIONED
# ============================================================
#
# The model only uses NORMALISED prototypes for cosine
# similarity, so prototype magnitude carries no predictive
# information.
#
# Re-projecting each prototype to unit norm therefore preserves
# its learned direction while avoiding extremely tiny vectors.
# ============================================================

@torch.no_grad()
def renormalise_prototypes(model):

    if not hasattr(
        model,
        "theme_prototypes"
    ):
        return

    p = (
        model
        .theme_prototypes
        .data
    )

    # Fail explicitly if the optimiser has already corrupted it.
    if not torch.isfinite(
        p
    ).all():

        raise FloatingPointError(
            "theme_prototypes became "
            "non-finite after optimizer.step()."
        )

    norms = torch.linalg.vector_norm(
        p,
        ord=2,
        dim=1,
        keepdim=True
    )

    # Avoid a zero vector.
    bad = (
        norms
        <
        STABLE_NORM_EPS
    )

    if bad.any():

        # Re-initialise only pathological near-zero prototypes.
        p[
            bad.squeeze(1)
        ] = (
            torch.randn_like(
                p[
                    bad.squeeze(1)
                ]
            )
            *
            0.02
        )

        norms = torch.linalg.vector_norm(
            p,
            ord=2,
            dim=1,
            keepdim=True
        )

    p.div_(
        norms.clamp(
            min=STABLE_NORM_EPS
        )
    )


# ============================================================
# 11F. FIND WHICH PARAMETER CREATED A BAD GRADIENT
# ============================================================

def find_bad_gradients(model):

    bad = []

    for name, param in (
        model.named_parameters()
    ):

        if (
            param.grad
            is not None
            and
            not torch.isfinite(
                param.grad
            ).all()
        ):

            bad.append(
                name
            )

    return bad


# ============================================================
# 11G. FINITE PREDICTION FUNCTION
# ============================================================

def predict_fusenet_finite(
    model,
    loader
):

    model.eval()

    bin_probs = []
    theme_probs = []
    gates = []


    with torch.no_grad():

        for batch in loader:

            bge = (
                batch[
                    "bge"
                ]
                .to(
                    DEVICE
                )
            )

            specter = (
                batch[
                    "specter"
                ]
                .to(
                    DEVICE
                )
            )

            tfidf = (
                batch[
                    "tfidf"
                ]
                .to(
                    DEVICE
                )
            )


            out = model(
                bge,
                specter,
                tfidf
            )


            # Do not fabricate probabilities.
            for key in [
                "binary_logits",
                "theme_logits",
                "gates"
            ]:

                if not torch.isfinite(
                    out[key]
                ).all():

                    raise FloatingPointError(
                        f"Non-finite {key} "
                        "during validation."
                    )


            bp = torch.sigmoid(
                out[
                    "binary_logits"
                ]
            )


            tp = torch.softmax(
                out[
                    "theme_logits"
                ],
                dim=1
            )


            gp = out[
                "gates"
            ]


            bin_probs.append(
                bp
                .cpu()
                .numpy()
            )

            theme_probs.append(
                tp
                .cpu()
                .numpy()
            )

            gates.append(
                gp
                .cpu()
                .numpy()
            )


    bp = (
        np.concatenate(
            bin_probs
        )
        .astype(
            np.float32
        )
    )

    tp = (
        np.vstack(
            theme_probs
        )
        .astype(
            np.float32
        )
    )

    gp = (
        np.vstack(
            gates
        )
        .astype(
            np.float32
        )
    )


    assert np.isfinite(
        bp
    ).all()

    assert np.isfinite(
        tp
    ).all()

    assert np.isfinite(
        gp
    ).all()


    return (
        bp,
        tp,
        gp
    )


# ============================================================
# 11H. STABLE SINGLE-FOLD TRAINING
# ============================================================

def train_fusenet_fold_stable(
    Xb,
    Xs,
    Xt,
    yb,
    yt,
    tr,
    va,
    fold_number
):

    # --------------------------------------------------------
    # Data
    # --------------------------------------------------------

    train_loader = (
        make_train_loader(
            Xb,
            Xs,
            Xt,
            yb,
            yt,
            tr
        )
    )


    val_loader = (
        make_eval_loader(
            Xb,
            Xs,
            Xt,
            yb,
            yt,
            va
        )
    )


    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    reset_all_seeds(
        SEED
        +
        fold_number
    )


    model = ClimateWellFuseNet(
        Xb.shape[1],
        Xs.shape[1],
        Xt.shape[1],
        len(
            theme_names
        )
    ).to(
        DEVICE
    )


    # Make initial prototype norms safe.
    renormalise_prototypes(
        model
    )


    # --------------------------------------------------------
    # Optimiser
    #
    # IMPORTANT:
    # Keep the dissertation hyperparameter:
    #
    # LR = 1.2e-3
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        eps=1e-6,
    )


    scheduler = (
        torch.optim
        .lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=MAX_EPOCHS
        )
    )


    # --------------------------------------------------------
    # Binary imbalance
    # --------------------------------------------------------

    pos = max(
        1,
        int(
            yb[
                tr
            ].sum()
        )
    )


    neg = max(
        1,
        int(
            len(
                tr
            )
            -
            yb[
                tr
            ].sum()
        )
    )


    pos_weight = torch.tensor(
        [
            math.sqrt(
                neg
                /
                pos
            )
        ],
        dtype=torch.float32,
        device=DEVICE
    )


    # --------------------------------------------------------
    # Theme weights
    # --------------------------------------------------------

    th_w = (
        theme_weights_for_idx(
            yt,
            tr
        )
    )


    # --------------------------------------------------------
    # Best finite checkpoint
    # --------------------------------------------------------

    best_score = -np.inf
    best_state = None
    best_epoch = 0

    bad_epochs = 0

    numerical_stop = False
    numerical_message = None


    target_rate = float(
        yb[
            tr
        ].mean()
    )


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(
        1,
        MAX_EPOCHS + 1
    ):

        model.train()

        epoch_loss = 0.0

        n_batches = 0


        # ----------------------------------------------------
        # MINI-BATCH LOOP
        # ----------------------------------------------------

        for batch_id, batch in enumerate(
            train_loader
        ):

            bge = (
                batch[
                    "bge"
                ]
                .to(
                    DEVICE
                )
            )

            specter = (
                batch[
                    "specter"
                ]
                .to(
                    DEVICE
                )
            )

            tfidf = (
                batch[
                    "tfidf"
                ]
                .to(
                    DEVICE
                )
            )

            ybb = (
                batch[
                    "y_bin"
                ]
                .to(
                    DEVICE
                )
            )

            ytt = (
                batch[
                    "y_theme"
                ]
                .to(
                    DEVICE
                )
            )


            # ------------------------------------------------
            # Forward
            # ------------------------------------------------

            out = model(
                bge,
                specter,
                tfidf
            )


            if not torch.isfinite(
                out[
                    "binary_logits"
                ]
            ).all():

                numerical_stop = True

                numerical_message = (
                    f"binary logits became "
                    f"non-finite at epoch "
                    f"{epoch}, batch "
                    f"{batch_id}"
                )

                break


            if not torch.isfinite(
                out[
                    "theme_logits"
                ]
            ).all():

                numerical_stop = True

                numerical_message = (
                    f"theme logits became "
                    f"non-finite at epoch "
                    f"{epoch}, batch "
                    f"{batch_id}"
                )

                break


            # ------------------------------------------------
            # Loss
            # ------------------------------------------------

            loss_bin = focal_bce(
                out[
                    "binary_logits"
                ],
                ybb,
                pos_weight
            )


            theme_mask = (
                ytt
                >=
                0
            )


            if theme_mask.any():

                loss_theme = focal_ce(
                    out[
                        "theme_logits"
                    ][
                        theme_mask
                    ],
                    ytt[
                        theme_mask
                    ],
                    th_w
                )

            else:

                loss_theme = (
                    torch.tensor(
                        0.0,
                        device=DEVICE
                    )
                )


            loss = (
                loss_bin

                +

                THEME_LOSS_WEIGHT
                *
                loss_theme

                +

                gate_reg(
                    out
                )
            )


            if not torch.isfinite(
                loss
            ):

                numerical_stop = True

                numerical_message = (
                    f"loss became non-finite "
                    f"at epoch {epoch}, "
                    f"batch {batch_id}"
                )

                break


            # ------------------------------------------------
            # Backward
            # ------------------------------------------------

            optimizer.zero_grad(
                set_to_none=True
            )


            loss.backward()


            # ----------------------------------------------
            # Detect the actual offending gradient BEFORE
            # global norm calculation.
            # ----------------------------------------------

            bad_grad_names = (
                find_bad_gradients(
                    model
                )
            )


            if bad_grad_names:

                numerical_stop = True

                numerical_message = (
                    f"non-finite gradient at "
                    f"epoch {epoch}, batch "
                    f"{batch_id}. "
                    f"Parameters: "
                    f"{bad_grad_names[:5]}"
                )

                break


            # ----------------------------------------------
            # Gradient clipping
            #
            # foreach=False avoids a fused foreach norm path
            # being another source of numeric overflow.
            # ----------------------------------------------

            grad_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0,
                    norm_type=2.0,
                    error_if_nonfinite=False,
                    foreach=False,
                )
            )


            if not torch.isfinite(
                grad_norm
            ):

                numerical_stop = True

                numerical_message = (
                    f"global gradient norm "
                    f"became non-finite at "
                    f"epoch {epoch}, "
                    f"batch {batch_id}"
                )

                break


            # ------------------------------------------------
            # Parameter update
            # ------------------------------------------------

            optimizer.step()


            # Keep prototypes well-conditioned.
            renormalise_prototypes(
                model
            )


            epoch_loss += float(
                loss
                .detach()
                .cpu()
            )


            n_batches += 1


        # ====================================================
        # IF NUMERICAL INSTABILITY OCCURS
        # ====================================================

        if numerical_stop:

            print(
                "\n⚠ Numerical early stop:"
            )

            print(
                numerical_message
            )


            if best_state is not None:

                print(
                    "A valid earlier checkpoint "
                    f"already exists from epoch "
                    f"{best_epoch}."
                )

                print(
                    "Stopping this fold and "
                    "restoring that best finite "
                    "checkpoint."
                )

                break


            # Instability before any valid validation checkpoint
            # is genuinely fatal.
            raise RuntimeError(
                "FuseNet became numerically "
                "unstable before producing a "
                "valid checkpoint.\n"
                +
                str(
                    numerical_message
                )
            )


        scheduler.step()


        # ====================================================
        # VALIDATION
        # ====================================================

        bp, tp, gp = (
            predict_fusenet_finite(
                model,
                val_loader
            )
        )


        thr_pack = tune_threshold(
            yb[
                va
            ],
            bp,
            target_rate
        )


        bm = binary_metrics(
            yb[
                va
            ],
            bp,
            thr_pack[
                "threshold"
            ]
        )


        valid = (
            yt[
                va
            ]
            >=
            0
        )


        if valid.any():

            tm = theme_metrics(
                yt[
                    va
                ][
                    valid
                ],
                tp[
                    valid
                ]
            )

        else:

            tm = {
                "theme_macro_f1":
                    0.0,

                "theme_weighted_f1":
                    0.0
            }


        score = (
            0.25
            *
            bm[
                "AP"
            ]

            +

            0.20
            *
            bm[
                "accept_f1"
            ]

            +

            0.15
            *
            bm[
                "binary_macro_f1"
            ]

            +

            0.35
            *
            tm[
                "theme_macro_f1"
            ]

            +

            0.05
            *
            tm[
                "theme_weighted_f1"
            ]
        )


        if (
            np.isfinite(
                score
            )
            and
            score
            >
            best_score
        ):

            best_score = float(
                score
            )


            best_epoch = int(
                epoch
            )


            best_state = {
                key:
                    value
                    .detach()
                    .cpu()
                    .clone()

                for (
                    key,
                    value
                )
                in
                model
                .state_dict()
                .items()
            }


            bad_epochs = 0


        else:

            bad_epochs += 1


        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------

        if (
            epoch == 1
            or
            epoch % 5 == 0
            or
            bad_epochs >= PATIENCE
        ):

            mean_loss = (
                epoch_loss
                /
                max(
                    n_batches,
                    1
                )
            )


            print(
                f"Fold {fold_number} | "
                f"epoch {epoch:02d} | "
                f"loss={mean_loss:.4f} | "
                f"AP={bm['AP']:.4f} | "
                f"Accept-F1="
                f"{bm['accept_f1']:.4f} | "
                f"Theme-mF1="
                f"{tm['theme_macro_f1']:.4f} | "
                f"score={score:.4f}"
            )


        # ----------------------------------------------------
        # Normal early stopping
        # ----------------------------------------------------

        if bad_epochs >= PATIENCE:

            print(
                f"Fold {fold_number}: "
                f"normal early stopping "
                f"at epoch {epoch}."
            )

            break


    # ========================================================
    # RESTORE BEST FINITE CHECKPOINT
    # ========================================================

    if best_state is None:

        raise RuntimeError(
            f"Fold {fold_number} produced "
            "no valid finite checkpoint."
        )


    model.load_state_dict(
        best_state
    )


    renormalise_prototypes(
        model
    )


    # Fresh loader just for deterministic final inference.
    val_loader = (
        make_eval_loader(
            Xb,
            Xs,
            Xt,
            yb,
            yt,
            va
        )
    )


    bp, tp, gp = (
        predict_fusenet_finite(
            model,
            val_loader
        )
    )


    return (
        model,
        bp,
        tp,
        gp,
        best_epoch,
        best_score
    )


# ============================================================
# 11I. RUN 5-FOLD OOF
# ============================================================

fuse_oof_bin = np.zeros(
    len(
        train_df
    ),
    dtype=np.float32
)


fuse_oof_theme = np.zeros(
    (
        len(
            train_df
        ),
        len(
            theme_names
        )
    ),
    dtype=np.float32
)


fuse_oof_gates = np.zeros(
    (
        len(
            train_df
        ),
        3
    ),
    dtype=np.float32
)


best_epochs = []


skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)


for fold, (
    tr,
    va
) in enumerate(
    skf.split(
        np.zeros(
            len(
                y_bin
            )
        ),
        y_bin
    ),
    1
):

    print("\n")
    print("=" * 80)

    print(
        f"FuseNet fold "
        f"{fold}/{N_FOLDS}"
    )

    print("=" * 80)


    # --------------------------------------------------------
    # Fold-safe TF-IDF / SVD
    # --------------------------------------------------------

    tf = fit_tfidf_svd(
        [
            train_texts[i]
            for i
            in tr
        ],
        [
            train_texts[i]
            for i
            in va
        ]
    )


    Xt = np.zeros(
        (
            len(
                train_df
            ),
            tf[
                "train"
            ].shape[
                1
            ]
        ),
        dtype=np.float32
    )


    Xt[
        tr
    ] = tf[
        "train"
    ].astype(
        np.float32
    )


    Xt[
        va
    ] = tf[
        "eval"
    ].astype(
        np.float32
    )


    # --------------------------------------------------------
    # BGE scaling
    # --------------------------------------------------------

    (
        Xb_tr,
        Xb_va,
        _
    ) = fit_dense_scaler(
        BGE_TRAIN[
            tr
        ],
        BGE_TRAIN[
            va
        ]
    )


    # --------------------------------------------------------
    # SPECTER scaling
    # --------------------------------------------------------

    (
        Xs_tr,
        Xs_va,
        _
    ) = fit_dense_scaler(
        SPECTER_TRAIN[
            tr
        ],
        SPECTER_TRAIN[
            va
        ]
    )


    Xb = np.zeros_like(
        BGE_TRAIN,
        dtype=np.float32
    )


    Xs = np.zeros_like(
        SPECTER_TRAIN,
        dtype=np.float32
    )


    Xb[
        tr
    ] = Xb_tr


    Xb[
        va
    ] = Xb_va


    Xs[
        tr
    ] = Xs_tr


    Xs[
        va
    ] = Xs_va


    # --------------------------------------------------------
    # Check transformed features
    # --------------------------------------------------------

    assert np.isfinite(
        Xb[
            tr
        ]
    ).all()


    assert np.isfinite(
        Xb[
            va
        ]
    ).all()


    assert np.isfinite(
        Xs[
            tr
        ]
    ).all()


    assert np.isfinite(
        Xs[
            va
        ]
    ).all()


    assert np.isfinite(
        Xt[
            tr
        ]
    ).all()


    assert np.isfinite(
        Xt[
            va
        ]
    ).all()


    # ========================================================
    # TRAIN
    # ========================================================

    (
        model,
        bp,
        tp,
        gp,
        be,
        bs
    ) = train_fusenet_fold_stable(
        Xb,
        Xs,
        Xt,
        y_bin,
        y_theme_full,
        tr,
        va,
        fold
    )


    # --------------------------------------------------------
    # OOF storage
    # --------------------------------------------------------

    fuse_oof_bin[
        va
    ] = bp


    fuse_oof_theme[
        va
    ] = tp


    fuse_oof_gates[
        va
    ] = gp


    best_epochs.append(
        be
    )


    # --------------------------------------------------------
    # Fold summary
    # --------------------------------------------------------

    fold_thr = tune_threshold(
        y_bin[
            va
        ],
        bp,
        TRAIN_ACCEPT_RATE
    )


    fold_bm = binary_metrics(
        y_bin[
            va
        ],
        bp,
        fold_thr[
            "threshold"
        ]
    )


    fold_valid_theme = (
        y_theme_full[
            va
        ]
        >=
        0
    )


    if fold_valid_theme.any():

        fold_tm = theme_metrics(
            y_theme_full[
                va
            ][
                fold_valid_theme
            ],
            tp[
                fold_valid_theme
            ]
        )

    else:

        fold_tm = {
            "theme_macro_f1":
                np.nan
        }


    print(
        "\nFold summary:"
    )


    print(
        {
            "fold":
                fold,

            "best_epoch":
                be,

            "threshold":
                round(
                    fold_thr[
                        "threshold"
                    ],
                    4
                ),

            "AP":
                round(
                    fold_bm[
                        "AP"
                    ],
                    4
                ),

            "binary_macro_f1":
                round(
                    fold_bm[
                        "binary_macro_f1"
                    ],
                    4
                ),

            "accept_f1":
                round(
                    fold_bm[
                        "accept_f1"
                    ],
                    4
                ),

            "theme_macro_f1":
                (
                    round(
                        fold_tm[
                            "theme_macro_f1"
                        ],
                        4
                    )
                    if
                    np.isfinite(
                        fold_tm[
                            "theme_macro_f1"
                        ]
                    )
                    else
                    np.nan
                )
        }
    )


    # --------------------------------------------------------
    # Memory cleanup
    # --------------------------------------------------------

    del (
        model,
        Xt,
        Xb,
        Xs,
        tf
    )


    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 11J. COMPLETE OOF INTEGRITY
# ============================================================

print("\n")
print("=" * 80)
print("FINAL OOF NUMERICAL CHECK")
print("=" * 80)


print(
    "Binary OOF finite:",
    np.isfinite(
        fuse_oof_bin
    ).all()
)


print(
    "Theme OOF finite:",
    np.isfinite(
        fuse_oof_theme
    ).all()
)


print(
    "Gate OOF finite:",
    np.isfinite(
        fuse_oof_gates
    ).all()
)


assert np.isfinite(
    fuse_oof_bin
).all()


assert np.isfinite(
    fuse_oof_theme
).all()


assert np.isfinite(
    fuse_oof_gates
).all()


# ============================================================
# 11K. FINAL THRESHOLD
# ============================================================

FUSENET_THRESHOLD = (
    tune_threshold(
        y_bin,
        fuse_oof_bin,
        TRAIN_ACCEPT_RATE
    )[
        "threshold"
    ]
)


# ============================================================
# 11L. FINAL STAGE-1 METRICS
# ============================================================

fusenet_stage1_internal = (
    binary_metrics(
        y_bin,
        fuse_oof_bin,
        FUSENET_THRESHOLD
    )
)


# ============================================================
# 11M. FINAL STAGE-2 METRICS
# ============================================================

valid_theme = (
    y_theme_full
    >=
    0
)


fusenet_theme_internal = (
    theme_metrics(
        y_theme_full[
            valid_theme
        ],
        fuse_oof_theme[
            valid_theme
        ]
    )
)


# ============================================================
# 11N. PRODUCTION EPOCH COUNT
# ============================================================

FINAL_FUSENET_EPOCHS = max(
    3,
    int(
        round(
            np.median(
                best_epochs
            )
        )
    )
)


print(
    "\nFold best epochs:",
    best_epochs
)


print(
    "Final production-fit epochs:",
    FINAL_FUSENET_EPOCHS
)


print(
    "Final FuseNet threshold:",
    round(
        FUSENET_THRESHOLD,
        4
    )
)


# ============================================================
# 11O. FINAL RESULTS
# ============================================================

display(
    pd.DataFrame(
        [
            {
                **fusenet_stage1_internal,
                **fusenet_theme_internal,

                "threshold":
                    FUSENET_THRESHOLD,

                "median_best_epoch":
                    FINAL_FUSENET_EPOCHS,
            }
        ]
    ).T
)


# ============================================================
# 11P. OVERRIDE FINAL FULL-DATA TRAINER FOR CELL 12
# ============================================================
#
# Cell 12 calls train_fusenet_all().
#
# Override it here so the production fit uses the same
# stable-normalisation / prototype-conditioning strategy.
# ============================================================

def train_fusenet_all(
    Xb,
    Xs,
    Xt,
    yb,
    yt,
    epochs
):

    reset_all_seeds(
        SEED
    )


    idx = np.arange(
        len(
            yb
        )
    )


    model = ClimateWellFuseNet(
        Xb.shape[1],
        Xs.shape[1],
        Xt.shape[1],
        len(
            theme_names
        )
    ).to(
        DEVICE
    )


    renormalise_prototypes(
        model
    )


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
        eps=1e-6,
    )


    scheduler = (
        torch.optim
        .lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=max(
                1,
                epochs
            )
        )
    )


    loader = (
        make_train_loader(
            Xb,
            Xs,
            Xt,
            yb,
            yt,
            idx
        )
    )


    pos = max(
        1,
        int(
            yb.sum()
        )
    )


    neg = max(
        1,
        int(
            len(
                yb
            )
            -
            yb.sum()
        )
    )


    pos_weight = torch.tensor(
        [
            math.sqrt(
                neg
                /
                pos
            )
        ],
        dtype=torch.float32,
        device=DEVICE
    )


    th_w = (
        theme_weights_for_idx(
            yt,
            idx
        )
    )


    last_finite_state = {
        key:
            value
            .detach()
            .cpu()
            .clone()

        for (
            key,
            value
        )
        in
        model
        .state_dict()
        .items()
    }


    for epoch in range(
        1,
        epochs + 1
    ):

        model.train()

        unstable = False


        for batch_id, batch in enumerate(
            loader
        ):

            out = model(
                batch[
                    "bge"
                ].to(
                    DEVICE
                ),

                batch[
                    "specter"
                ].to(
                    DEVICE
                ),

                batch[
                    "tfidf"
                ].to(
                    DEVICE
                )
            )


            ybb = (
                batch[
                    "y_bin"
                ]
                .to(
                    DEVICE
                )
            )


            ytt = (
                batch[
                    "y_theme"
                ]
                .to(
                    DEVICE
                )
            )


            lb = focal_bce(
                out[
                    "binary_logits"
                ],
                ybb,
                pos_weight
            )


            mask = (
                ytt
                >=
                0
            )


            if mask.any():

                lt = focal_ce(
                    out[
                        "theme_logits"
                    ][
                        mask
                    ],
                    ytt[
                        mask
                    ],
                    th_w
                )

            else:

                lt = torch.tensor(
                    0.0,
                    device=DEVICE
                )


            loss = (
                lb

                +

                THEME_LOSS_WEIGHT
                *
                lt

                +

                gate_reg(
                    out
                )
            )


            if not torch.isfinite(
                loss
            ):

                unstable = True

                print(
                    f"Production fit: "
                    f"non-finite loss at "
                    f"epoch {epoch}, "
                    f"batch {batch_id}."
                )

                break


            optimizer.zero_grad(
                set_to_none=True
            )


            loss.backward()


            bad_grads = (
                find_bad_gradients(
                    model
                )
            )


            if bad_grads:

                unstable = True

                print(
                    f"Production fit: "
                    f"non-finite gradient at "
                    f"epoch {epoch}, "
                    f"batch {batch_id}: "
                    f"{bad_grads[:5]}"
                )

                break


            grad_norm = (
                torch.nn.utils
                .clip_grad_norm_(
                    model.parameters(),
                    1.0,
                    error_if_nonfinite=False,
                    foreach=False,
                )
            )


            if not torch.isfinite(
                grad_norm
            ):

                unstable = True

                print(
                    f"Production fit: "
                    "non-finite gradient norm. "
                    "Restoring last finite epoch."
                )

                break


            optimizer.step()


            renormalise_prototypes(
                model
            )


        if unstable:

            model.load_state_dict(
                last_finite_state
            )

            renormalise_prototypes(
                model
            )

            print(
                "Production training stopped "
                "at the last finite checkpoint."
            )

            break


        # Save a complete finite epoch.
        last_finite_state = {
            key:
                value
                .detach()
                .cpu()
                .clone()

            for (
                key,
                value
            )
            in
            model
            .state_dict()
            .items()
        }


        scheduler.step()


        if (
            epoch == 1
            or
            epoch % 5 == 0
            or
            epoch == epochs
        ):

            print(
                f"Final FuseNet fit: "
                f"epoch {epoch}/{epochs}"
            )


    model.load_state_dict(
        last_finite_state
    )


    renormalise_prototypes(
        model
    )


    return model


print(
    "\n✓ Stable train_fusenet_all() installed "
    "for the next cell."
)

PRE-FUSENET NUMERICAL CHECK
BGE_TRAIN                 shape=(1720, 384)        NaN=0       +Inf=0       -Inf=0      
SPECTER_TRAIN             shape=(1720, 768)        NaN=0       +Inf=0       -Inf=0      

✓ Patched ClimateWellFuseNet.forward with stable prototype normalisation.


FuseNet fold 1/5
Fold 1 | epoch 01 | loss=2.5956 | AP=0.3253 | Accept-F1=0.3902 | Theme-mF1=0.1178 | score=0.3180
Fold 1 | epoch 05 | loss=0.2211 | AP=0.3326 | Accept-F1=0.3908 | Theme-mF1=0.1128 | score=0.3169
Fold 1 | epoch 10 | loss=0.1053 | AP=0.3431 | Accept-F1=0.4051 | Theme-mF1=0.1266 | score=0.3297
Fold 1 | epoch 12 | loss=0.1400 | AP=0.4220 | Accept-F1=0.4156 | Theme-mF1=0.0919 | score=0.3347
Fold 1: normal early stopping at epoch 12.

Fold summary:
{'fold': 1, 'best_epoch': 2, 'threshold': 0.735, 'AP': 0.5022, 'binary_macro_f1': 0.7277, 'accept_f1': 0.5275, 'theme_macro_f1': 0.099}


FuseNet fold 2/5
Fold 2 | epoch 01 | loss=2.7181 | AP=0.3273 | Accept-F1=0.3704 | Theme-mF1=0.0921 | score=0.3006
Fo

,0
AP,0.364914
binary_macro_f1,0.671073
accept_precision,0.422680
accept_recall,0.412060
accept_f1,0.417303
pred_accept_rate,0.112791
tn,1409.000000
fp,112.000000
fn,117.000000
tp,82.000000



✓ Stable train_fusenet_all() installed for the next cell.


In [16]:
# ============================================================
# 12. FIT FINAL FUSENET ON ALL TRAINING DATA + AUDIT PREPROCESSORS
# ============================================================
# Fit every preprocessing transform on labelled training only.
FUSE_TF = fit_tfidf_svd(train_texts, audit_texts)
FUSE_XT_TRAIN = FUSE_TF["train"]
FUSE_XT_AUDIT = FUSE_TF["eval"]

FUSE_XB_TRAIN,FUSE_XB_AUDIT,FUSE_BGE_SCALER = fit_dense_scaler(BGE_TRAIN,BGE_AUDIT)
FUSE_XS_TRAIN,FUSE_XS_AUDIT,FUSE_SPECTER_SCALER = fit_dense_scaler(SPECTER_TRAIN,SPECTER_AUDIT)

FUSENET_MODEL = train_fusenet_all(
    FUSE_XB_TRAIN,FUSE_XS_TRAIN,FUSE_XT_TRAIN,
    y_bin,y_theme_full,FINAL_FUSENET_EPOCHS
)

print("Final FuseNet model fitted.")

Final FuseNet fit: epoch 1/7
Final FuseNet fit: epoch 5/7
Final FuseNet fit: epoch 7/7
Final FuseNet model fitted.


In [17]:
# ============================================================
# 13. AUDIT PREDICTIONS FOR BOTH MODELS
# ============================================================
# These predictions are always generated. Metrics are reported only when
# AUDIT_LABELS_VERIFIED=True.

# Model A: BGE + TF-IDF
cl_audit_p, cl_audit_d, cl_audit_theme_p = classical_predict_batch(
    audit_texts, BGE_AUDIT
)

# Model B: FuseNet V2
audit_ds = ClimateDataset(
    FUSE_XB_AUDIT,FUSE_XS_AUDIT,FUSE_XT_AUDIT,
    np.zeros(len(audit_df),dtype=np.float32),
    -np.ones(len(audit_df),dtype=int)
)
audit_loader = DataLoader(audit_ds,batch_size=BATCH_SIZE*2,shuffle=False)
fu_audit_p,fu_audit_theme_p,fu_audit_gates = predict_fusenet_loader(FUSENET_MODEL,audit_loader)
fu_audit_d = (fu_audit_p >= FUSENET_THRESHOLD).astype(int)

audit_predictions = audit_df[["paper_id","title","abstract"]].copy()
audit_predictions["bge_tfidf_p_accept"] = cl_audit_p
audit_predictions["bge_tfidf_decision"] = np.where(cl_audit_d==1,"Accept","Reject")
audit_predictions["bge_tfidf_theme"] = [id_to_theme[int(i)] for i in cl_audit_theme_p.argmax(1)]
audit_predictions["fusenet_p_accept"] = fu_audit_p
audit_predictions["fusenet_decision"] = np.where(fu_audit_d==1,"Accept","Reject")
audit_predictions["fusenet_theme"] = [id_to_theme[int(i)] for i in fu_audit_theme_p.argmax(1)]
audit_predictions["gate_bge"] = fu_audit_gates[:,0]
audit_predictions["gate_specter"] = fu_audit_gates[:,1]
audit_predictions["gate_tfidf"] = fu_audit_gates[:,2]

external_rows = []
if AUDIT_LABELS_VERIFIED:
    y_audit = audit_df["audit_label"].astype(int).values
    valid_audit_theme = (
        audit_df["audit_label"].eq(1)
        & audit_df["audit_theme_id"].notna()
    ).values
    y_audit_theme = audit_df.loc[valid_audit_theme,"audit_theme_id"].astype(int).values

    cl_bin = binary_metrics(y_audit,cl_audit_p,CLASSICAL_THRESHOLD)
    fu_bin = binary_metrics(y_audit,fu_audit_p,FUSENET_THRESHOLD)
    cl_th = theme_metrics(y_audit_theme,cl_audit_theme_p[valid_audit_theme])
    fu_th = theme_metrics(y_audit_theme,fu_audit_theme_p[valid_audit_theme])

    external_rows = [
        {"model":"BGE+TF-IDF",**cl_bin,**cl_th},
        {"model":"ClimateWell-FuseNet V2",**fu_bin,**fu_th},
    ]
    display(pd.DataFrame(external_rows))
else:
    print("External audit metrics skipped because AUDIT_LABELS_VERIFIED=False.")
    print("Predictions are still saved for inspection; no ground-truth claim is made.")

audit_predictions.to_csv(OUTPUT_DIR/"audit_model_predictions.csv",index=False)

External audit metrics skipped because AUDIT_LABELS_VERIFIED=False.
Predictions are still saved for inspection; no ground-truth claim is made.


In [18]:
# ============================================================
# 14. HEAD-TO-HEAD MODEL COMPARISON + PRIMARY MODEL SELECTION
# ============================================================
comparison = pd.DataFrame([
    {
        "model":"BGE+TF-IDF",
        "internal_AP":classical_stage1_internal["AP"],
        "internal_binary_macro_f1":classical_stage1_internal["binary_macro_f1"],
        "internal_accept_f1":classical_stage1_internal["accept_f1"],
        "internal_theme_macro_f1":classical_theme_internal["theme_macro_f1"],
    },
    {
        "model":"ClimateWell-FuseNet V2",
        "internal_AP":fusenet_stage1_internal["AP"],
        "internal_binary_macro_f1":fusenet_stage1_internal["binary_macro_f1"],
        "internal_accept_f1":fusenet_stage1_internal["accept_f1"],
        "internal_theme_macro_f1":fusenet_theme_internal["theme_macro_f1"],
    }
])

if AUDIT_LABELS_VERIFIED:
    ext = pd.DataFrame(external_rows)[[
        "model","AP","accept_f1","theme_macro_f1","pred_accept_rate"
    ]].rename(columns={
        "AP":"external_AP",
        "accept_f1":"external_accept_f1",
        "theme_macro_f1":"external_theme_macro_f1",
        "pred_accept_rate":"external_pred_accept_rate"
    })
    comparison = comparison.merge(ext,on="model",how="left")

    # This score is ONLY a transparent prototype selection rule; it is not
    # reported as a dissertation metric.
    comparison["prototype_selection_score"] = (
        0.40*comparison["external_AP"]
        + 0.30*comparison["external_accept_f1"]
        + 0.30*comparison["external_theme_macro_f1"]
    )
    PRIMARY_MODEL = comparison.sort_values(
        "prototype_selection_score",ascending=False
    ).iloc[0]["model"]
    SELECTION_REASON = (
        "Selected automatically from verified external audit using "
        "40% AP + 30% Accept-F1 + 30% theme macro-F1."
    )
else:
    # This is the final model-selection policy documented in the dissertation.
    PRIMARY_MODEL = "BGE+TF-IDF"
    comparison["prototype_selection_score"] = np.nan
    SELECTION_REASON = (
        "Audit columns not verified as human ground truth; therefore the notebook "
        "uses the dissertation's final deployment recommendation: BGE+TF-IDF as "
        "primary system and FuseNet V2 as challenger/uncertainty model."
    )

CHALLENGER_MODEL = (
    "ClimateWell-FuseNet V2" if PRIMARY_MODEL=="BGE+TF-IDF" else "BGE+TF-IDF"
)

print("PRIMARY MODEL:",PRIMARY_MODEL)
print("CHALLENGER:",CHALLENGER_MODEL)
print(SELECTION_REASON)
display(comparison)

comparison.to_csv(OUTPUT_DIR/"model_comparison.csv",index=False)

# Dissertation reference numbers for sanity checking only.
reference = pd.DataFrame([
    {
        "model":"BGE+TF-IDF (retained classical)",
        "reported_internal_AP":0.4036,
        "reported_internal_binary_macro_f1":0.6928,
        "reported_internal_accept_f1":0.4581,
        "reported_internal_theme_macro_f1":0.3139,
        "reported_external_AP":0.7929,
        "reported_external_accept_f1":0.7523,
    },
    {
        "model":"ClimateWell-FuseNet V2",
        "reported_internal_AP":0.4184,
        "reported_internal_binary_macro_f1":0.7055,
        "reported_internal_accept_f1":0.4802,
        "reported_internal_theme_macro_f1":0.2157,
        "reported_external_AP":np.nan,
        "reported_external_accept_f1":np.nan,
    }
])
print("\nReference values reported in the final dissertation (not recomputed here):")
display(reference)

PRIMARY MODEL: BGE+TF-IDF
CHALLENGER: ClimateWell-FuseNet V2
Audit columns not verified as human ground truth; therefore the notebook uses the dissertation's final deployment recommendation: BGE+TF-IDF as primary system and FuseNet V2 as challenger/uncertainty model.


,model,internal_AP,internal_binary_macro_f1,internal_accept_f1,internal_theme_macro_f1,prototype_selection_score
0,BGE+TF-IDF,0.427884,0.713215,0.490956,0.306614,NaN
1,ClimateWell-FuseNet V2,0.364914,0.671073,0.417303,0.225735,NaN



Reference values reported in the final dissertation (not recomputed here):


,model,reported_internal_AP,reported_internal_binary_macro_f1,reported_internal_accept_f1,reported_internal_theme_macro_f1,reported_external_AP,reported_external_accept_f1
0,BGE+TF-IDF (retained classical),0.4036,0.6928,0.4581,0.3139,0.7929,0.7523
1,ClimateWell-FuseNet V2,0.4184,0.7055,0.4802,0.2157,NaN,NaN


In [19]:
# ============================================================
# 15. MASTER-CORPUS INFERENCE FOR BOTH MODELS
#     + GENUINELY UNSEEN DEFENCE EXAMPLES
# ============================================================
def fuse_transform_texts(texts, bge_emb, specter_emb):
    Xtf_sparse = sp.hstack([
        FUSE_TF["word_vec"].transform(texts),
        FUSE_TF["char_vec"].transform(texts)
    ]).tocsr()
    Xt = FUSE_TF["scaler"].transform(
        FUSE_TF["svd"].transform(Xtf_sparse)
    ).astype(np.float32)
    Xb = FUSE_BGE_SCALER.transform(bge_emb).astype(np.float32)
    Xs = FUSE_SPECTER_SCALER.transform(specter_emb).astype(np.float32)
    return Xb,Xs,Xt

def fusenet_predict_batch(texts,bge_emb,specter_emb):
    Xb,Xs,Xt = fuse_transform_texts(texts,bge_emb,specter_emb)
    ds = ClimateDataset(
        Xb,Xs,Xt,
        np.zeros(len(texts),dtype=np.float32),
        -np.ones(len(texts),dtype=int)
    )
    loader = DataLoader(ds,batch_size=BATCH_SIZE*2,shuffle=False)
    bp,tp,gp = predict_fusenet_loader(FUSENET_MODEL,loader)
    d = (bp >= FUSENET_THRESHOLD).astype(int)
    return bp,d,tp,gp

if RUN_FULL_MASTER_INFERENCE:
    print("Predicting full master corpus with both candidate models...")
    cl_mp,cl_md,cl_mt = classical_predict_batch(master_texts,BGE_MASTER)
    fu_mp,fu_md,fu_mt,fu_mg = fusenet_predict_batch(
        master_texts,BGE_MASTER,SPECTER_MASTER
    )

    pred_master = master_df[[
        "paper_id","norm_id","title","abstract","ID_OLD","ID_New"
    ]].copy()
    pred_master["bge_tfidf_p_accept"] = cl_mp
    pred_master["bge_tfidf_decision"] = np.where(cl_md==1,"Accept","Reject")
    pred_master["bge_tfidf_theme"] = [id_to_theme[int(i)] for i in cl_mt.argmax(1)]
    pred_master["bge_tfidf_theme_conf"] = cl_mt.max(1)

    pred_master["fusenet_p_accept"] = fu_mp
    pred_master["fusenet_decision"] = np.where(fu_md==1,"Accept","Reject")
    pred_master["fusenet_theme"] = [id_to_theme[int(i)] for i in fu_mt.argmax(1)]
    pred_master["fusenet_theme_conf"] = fu_mt.max(1)
    pred_master["gate_bge"] = fu_mg[:,0]
    pred_master["gate_specter"] = fu_mg[:,1]
    pred_master["gate_tfidf"] = fu_mg[:,2]

    pred_master["is_training_record"] = pred_master["norm_id"].isin(train_ids)
    pred_master["is_audit_record"] = pred_master["norm_id"].isin(audit_ids)
    pred_master["eligible_unseen_demo"] = ~(
        pred_master["is_training_record"] | pred_master["is_audit_record"]
    )

    unseen_predictions = pred_master[pred_master["eligible_unseen_demo"]].copy()
    assert len(unseen_predictions)==7956

    # Primary-model convenience fields
    if PRIMARY_MODEL=="BGE+TF-IDF":
        unseen_predictions["primary_p_accept"] = unseen_predictions["bge_tfidf_p_accept"]
        unseen_predictions["primary_decision"] = unseen_predictions["bge_tfidf_decision"]
        unseen_predictions["primary_theme"] = unseen_predictions["bge_tfidf_theme"]
        unseen_predictions["primary_theme_conf"] = unseen_predictions["bge_tfidf_theme_conf"]
        unseen_predictions["challenger_decision"] = unseen_predictions["fusenet_decision"]
    else:
        unseen_predictions["primary_p_accept"] = unseen_predictions["fusenet_p_accept"]
        unseen_predictions["primary_decision"] = unseen_predictions["fusenet_decision"]
        unseen_predictions["primary_theme"] = unseen_predictions["fusenet_theme"]
        unseen_predictions["primary_theme_conf"] = unseen_predictions["fusenet_theme_conf"]
        unseen_predictions["challenger_decision"] = unseen_predictions["bge_tfidf_decision"]

    unseen_predictions["model_disagreement"] = (
        unseen_predictions["bge_tfidf_decision"] != unseen_predictions["fusenet_decision"]
    )

    # Curated genuine examples for the defence
    examples = []
    acc = unseen_predictions[unseen_predictions.primary_decision.eq("Accept")].sort_values(
        ["primary_p_accept","primary_theme_conf"],ascending=False
    )
    rej = unseen_predictions[unseen_predictions.primary_decision.eq("Reject")].sort_values(
        "primary_p_accept",ascending=True
    )
    dis = unseen_predictions[unseen_predictions.model_disagreement].copy()
    if len(acc): examples.append(("High-confidence ACCEPT",acc.iloc[0]))
    if len(rej): examples.append(("High-confidence REJECT",rej.iloc[0]))
    if len(dis):
        dis["boundary"] = (dis["primary_p_accept"]-0.5).abs()
        examples.append(("Model disagreement",dis.sort_values("boundary").iloc[0]))

    defence_examples = pd.DataFrame([
        {"example_type":kind, **row.to_dict()} for kind,row in examples
    ])

    unseen_predictions.to_csv(OUTPUT_DIR/"unseen_master_predictions.csv",index=False)
    defence_examples.to_csv(OUTPUT_DIR/"defence_examples.csv",index=False)

    print("Unseen eligible records:",len(unseen_predictions))
    print("Primary predicted Accept rate on unseen pool:",
          (unseen_predictions.primary_decision=="Accept").mean())
    print("Model disagreements:",unseen_predictions.model_disagreement.sum())
    display(defence_examples[[
        "example_type","paper_id","title","primary_decision",
        "primary_p_accept","primary_theme","model_disagreement"
    ]])
else:
    unseen_predictions = unseen_master.copy()
    defence_examples = pd.DataFrame()
    print("Full master inference skipped. Gradio will still perform live inference.")

Predicting full master corpus with both candidate models...
Unseen eligible records: 7956
Primary predicted Accept rate on unseen pool: 0.09728506787330317
Model disagreements: 544


,example_type,paper_id,title,primary_decision,primary_p_accept,primary_theme,model_disagreement
0,High-confidence ACCEPT,Scopus_0744,"Economic Growth, Climate Change and Clean Ener...",Accept,0.750000,SDGs and Well-being,False
1,High-confidence REJECT,WoS_5214,Potentially toxic metals in road dust: An asse...,Reject,0.000000,Climate change mitigation options & Well-being,False
2,Model disagreement,OA_5136,Acacia Shrubs and Trees for Climate Change Mit...,Accept,0.547619,Climate change mitigation options & Well-being,True


In [20]:
# ============================================================
# 16. SAVE REUSABLE MODEL ARTIFACTS
# ============================================================
if SAVE_MODEL_ARTIFACTS:
    classical_bundle = {
        "stage1_word_vec":CL_WV1,
        "stage1_char_vec":CL_CV1,
        "stage1_models":CL_STAGE1_MODELS,
        "calibrator":classical_calibrator,
        "threshold":CLASSICAL_THRESHOLD,
        "stage2_word_vec":CL_WV2,
        "stage2_char_vec":CL_CV2,
        "stage2_models":CL_STAGE2_MODELS,
        "theme_bge":CL_THEME_BGE,
        "theme_y":CL_THEME_Y,
        "theme_counts":CL_THEME_COUNTS,
        "theme_prior_alpha":CLASSICAL_THEME_ALPHA,
        "theme_names":theme_names,
    }
    joblib.dump(classical_bundle,ARTIFACT_DIR/"bge_tfidf_classical.joblib")

    fuse_preproc = {
        "word_vec":FUSE_TF["word_vec"],
        "char_vec":FUSE_TF["char_vec"],
        "svd":FUSE_TF["svd"],
        "tfidf_scaler":FUSE_TF["scaler"],
        "bge_scaler":FUSE_BGE_SCALER,
        "specter_scaler":FUSE_SPECTER_SCALER,
        "threshold":FUSENET_THRESHOLD,
        "theme_names":theme_names,
        "epochs":FINAL_FUSENET_EPOCHS,
    }
    joblib.dump(fuse_preproc,ARTIFACT_DIR/"fusenet_preprocessors.joblib")
    torch.save(
        {
            "state_dict":FUSENET_MODEL.state_dict(),
            "bge_dim":BGE_TRAIN.shape[1],
            "specter_dim":SPECTER_TRAIN.shape[1],
            "tfidf_dim":FUSE_XT_TRAIN.shape[1],
            "n_themes":len(theme_names),
        },
        ARTIFACT_DIR/"climatewell_fusenet_v2.pt"
    )

    with open(ARTIFACT_DIR/"deployment_policy.json","w") as f:
        json.dump({
            "primary_model":PRIMARY_MODEL,
            "challenger_model":CHALLENGER_MODEL,
            "selection_reason":SELECTION_REASON,
            "audit_labels_verified":AUDIT_LABELS_VERIFIED,
            "classical_threshold":float(CLASSICAL_THRESHOLD),
            "fusenet_threshold":float(FUSENET_THRESHOLD),
            "theme_names":theme_names,
        },f,indent=2)

    print("Artifacts saved to:",ARTIFACT_DIR)

Artifacts saved to: /kaggle/working/climatewell_defence_demo/artifacts


# Real-time Gradio demonstration

The interface below performs **fresh inference from the title and abstract fields**.

- **Primary model**: selected by verified audit comparison when available; otherwise the dissertation deployment policy.
- **Challenger model**: the other model.
- A theme is shown **only when the corresponding model predicts Accept**.
- Model disagreement or low confidence triggers higher human-review priority.
- The “Load …” buttons draw only from the **7,956 master records excluded from both training and audit**.

In [21]:
# ============================================================
# 17. LIVE ENCODERS FOR THE GRADIO APP
# ============================================================
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel

print("Loading live BGE encoder...")
LIVE_BGE = SentenceTransformer(BGE_MODEL_NAME,device=DEVICE)

print("Loading live SPECTER encoder...")
LIVE_SPECTER_TOKENIZER = AutoTokenizer.from_pretrained(SPECTER_MODEL_NAME)
LIVE_SPECTER = AutoModel.from_pretrained(SPECTER_MODEL_NAME).to(DEVICE).eval()

def encode_bge_live(text):
    return LIVE_BGE.encode(
        [text], normalize_embeddings=True, convert_to_numpy=True
    ).astype(np.float32)

def encode_specter_live(text):
    enc = LIVE_SPECTER_TOKENIZER(
        [text],padding=True,truncation=True,max_length=512,return_tensors="pt"
    )
    enc = {k:v.to(DEVICE) for k,v in enc.items()}
    with torch.no_grad():
        out = LIVE_SPECTER(**enc)
        z = F.normalize(out.last_hidden_state[:,0,:],p=2,dim=1)
    return z.cpu().numpy().astype(np.float32)

def predict_one_classical(text,bge):
    p,d,tp = classical_predict_batch([text],bge)
    tid = int(tp.argmax(1)[0])
    return {
        "model":"BGE+TF-IDF",
        "p_accept":float(p[0]),
        "threshold":float(CLASSICAL_THRESHOLD),
        "decision":"Accept" if int(d[0])==1 else "Reject",
        "theme":id_to_theme[tid],
        "theme_conf":float(tp[0,tid]),
    }

def predict_one_fusenet(text,bge,specter):
    p,d,tp,g = fusenet_predict_batch([text],bge,specter)
    tid = int(tp.argmax(1)[0])
    return {
        "model":"ClimateWell-FuseNet V2",
        "p_accept":float(p[0]),
        "threshold":float(FUSENET_THRESHOLD),
        "decision":"Accept" if int(d[0])==1 else "Reject",
        "theme":id_to_theme[tid],
        "theme_conf":float(tp[0,tid]),
        "gate_bge":float(g[0,0]),
        "gate_specter":float(g[0,1]),
        "gate_tfidf":float(g[0,2]),
    }

print("Live inference encoders ready.")

Loading live BGE encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading live SPECTER encoder...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Live inference encoders ready.


In [22]:
# ============================================================
# 18. GRADIO PROTOTYPE
# ============================================================
import gradio as gr

def choose_example(kind):
    if RUN_FULL_MASTER_INFERENCE and len(defence_examples):
        if kind != "Random unseen":
            hit = defence_examples[defence_examples.example_type.eq(kind)]
            if len(hit):
                r = hit.iloc[0]
                return str(r.paper_id),str(r.title),str(r.abstract)
        r = unseen_predictions.sample(1,random_state=random.randint(0,10_000_000)).iloc[0]
    else:
        r = unseen_master.sample(1,random_state=random.randint(0,10_000_000)).iloc[0]
    return str(r.paper_id),str(r.title),str(r.abstract)

def render_model_result(r,is_primary=False):
    heading = "PRIMARY DEPLOYMENT MODEL" if is_primary else "CHALLENGER MODEL"
    theme_line = (
        f"**Theme:** {r['theme']}  \n**Theme confidence:** {r['theme_conf']:.3f}"
        if r["decision"]=="Accept"
        else "**Theme:** Not applicable — Stage 2 is not invoked for rejected records."
    )
    return (
        f"### {heading}: {r['model']}\n"
        f"**Decision:** {r['decision']}  \n"
        f"**P(Accept):** {r['p_accept']:.3f}  \n"
        f"**Decision threshold:** {r['threshold']:.3f}  \n"
        f"{theme_line}"
    )

def analyse_record(record_id,title,abstract):
    title,abstract = clean_text(title),clean_text(abstract)
    text = build_text(title,abstract)
    if len(text)<30:
        return "Please provide a title and abstract.","","",pd.DataFrame()

    bge = encode_bge_live(text)
    specter = encode_specter_live(text)

    cl = predict_one_classical(text,bge)
    fu = predict_one_fusenet(text,bge,specter)

    primary = cl if PRIMARY_MODEL=="BGE+TF-IDF" else fu
    challenger = fu if PRIMARY_MODEL=="BGE+TF-IDF" else cl

    agree_decision = primary["decision"]==challenger["decision"]
    agree_theme = (
        primary["theme"]==challenger["theme"]
        if primary["decision"]=="Accept" and challenger["decision"]=="Accept"
        else None
    )

    near_boundary = abs(primary["p_accept"]-primary["threshold"])<0.05
    low_theme_conf = primary["decision"]=="Accept" and primary["theme_conf"]<0.40
    if not agree_decision or (agree_theme is False):
        priority="HIGH — model disagreement"
    elif near_boundary or low_theme_conf:
        priority="MEDIUM — borderline / low confidence"
    else:
        priority="LOW — models broadly stable"

    eligibility = "Custom input"
    nid = norm_id(record_id)
    if nid:
        if nid in train_ids:
            eligibility="WARNING: this ID belongs to the labelled training set"
        elif nid in audit_ids:
            eligibility="WARNING: this ID belongs to the audit set"
        elif nid in master_ids:
            eligibility="Verified unseen master-corpus record"

    agreement = (
        f"### Human-review signal\n"
        f"**Review priority:** {priority}  \n"
        f"**Decision agreement:** {'Yes' if agree_decision else 'No'}  \n"
        f"**Theme agreement:** "
        f"{'Yes' if agree_theme is True else 'No' if agree_theme is False else 'N/A'}  \n"
        f"**Record status:** {eligibility}"
    )

    gates = pd.DataFrame({
        "FuseNet modality":["BGE","SPECTER","TF-IDF/SVD"],
        "gate weight":[fu["gate_bge"],fu["gate_specter"],fu["gate_tfidf"]]
    })

    return (
        render_model_result(primary,True),
        render_model_result(challenger,False),
        agreement,
        gates
    )

with gr.Blocks(title="ClimateWell Defence Demo") as demo:
    gr.Markdown(
        "# ClimateWell — Real-Time Evidence-Mapping Demo\n"
        f"**Primary:** {PRIMARY_MODEL} · **Challenger:** {CHALLENGER_MODEL}\n\n"
        "Enter a climate-policy article title and abstract, or load a genuine unseen master-corpus example."
    )

    with gr.Row():
        example_kind = gr.Dropdown(
            choices=["Random unseen","High-confidence ACCEPT","High-confidence REJECT","Model disagreement"],
            value="Random unseen",label="Example type"
        )
        load_btn = gr.Button("Load Unseen Master Record",variant="secondary")

    record_id = gr.Textbox(label="Record ID",interactive=True)
    title_box = gr.Textbox(label="Article title",lines=2)
    abstract_box = gr.Textbox(label="Abstract",lines=9)
    analyse_btn = gr.Button("Analyse Record",variant="primary")

    with gr.Row():
        primary_out = gr.Markdown()
        challenger_out = gr.Markdown()
    agreement_out = gr.Markdown()
    gate_out = gr.Dataframe(
        headers=["FuseNet modality","gate weight"],
        label="ClimateWell-FuseNet V2 modality gates",
        interactive=False
    )

    load_btn.click(
        fn=choose_example,
        inputs=[example_kind],
        outputs=[record_id,title_box,abstract_box]
    )
    analyse_btn.click(
        fn=analyse_record,
        inputs=[record_id,title_box,abstract_box],
        outputs=[primary_out,challenger_out,agreement_out,gate_out]
    )

demo.launch(share=LAUNCH_GRADIO_SHARE_LINK)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://bd561cd60577ddcc52.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Defence script for the demo

A concise explanation while the interface is running:

> “This is not a fabricated example. The interface loads a title and abstract from the master corpus after excluding every record used in labelled development and the independent audit. The primary model first estimates well-being relevance. Only if the record passes the frozen screening threshold is Stage 2 invoked to assign one of the 12 themes. In parallel, ClimateWell-FuseNet V2 produces a challenger decision and instance-specific modality weights. A disagreement or borderline probability is surfaced for human review rather than being hidden.”

For a live defence, use:
1. **High-confidence ACCEPT** — demonstrates both stages.
2. **High-confidence REJECT** — demonstrates that rejected records receive no artificial theme.
3. **Model disagreement** — demonstrates the human-in-the-loop auditing contribution.

In [23]:
# ============================================================
# 19. EXPORT ALL DEFENCE OUTPUTS AS ONE ZIP
# ============================================================
# Save a compact summary before zipping.
summary = {
    "primary_model":PRIMARY_MODEL,
    "challenger_model":CHALLENGER_MODEL,
    "selection_reason":SELECTION_REASON,
    "audit_labels_verified":AUDIT_LABELS_VERIFIED,
    "n_training":len(train_df),
    "n_master":len(master_df),
    "n_audit":len(audit_df),
    "n_unseen_demo_pool":len(unseen_master),
    "classical_threshold":float(CLASSICAL_THRESHOLD),
    "fusenet_threshold":float(FUSENET_THRESHOLD),
    "classical_internal":{**classical_stage1_internal,**classical_theme_internal},
    "fusenet_internal":{**fusenet_stage1_internal,**fusenet_theme_internal},
}
with open(OUTPUT_DIR/"run_summary.json","w") as f:
    json.dump(summary,f,indent=2)

zip_out = ROOT/"ClimateWell_Defence_Demo_Outputs.zip"
with zipfile.ZipFile(zip_out,"w",zipfile.ZIP_DEFLATED) as z:
    for p in OUTPUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p,arcname=p.relative_to(OUTPUT_DIR.parent))

print("All outputs zipped to:",zip_out)
print("For Kaggle: download it from the Files panel under /kaggle/working.")
print("For Colab: run the optional cell below to download directly.")

All outputs zipped to: /kaggle/working/ClimateWell_Defence_Demo_Outputs.zip
For Kaggle: download it from the Files panel under /kaggle/working.
For Colab: run the optional cell below to download directly.


In [ ]:
# OPTIONAL — COLAB DIRECT DOWNLOAD
if IS_COLAB:
    from google.colab import files
    files.download(str(ROOT/"ClimateWell_Defence_Demo_Outputs.zip"))
else:
    print("Not running in Colab; use the Kaggle Files panel or notebook file browser.")